# Exécution sur Colab — évaluer, corriger, puis spécialiser

Après deux interruptions de l'évaluation locale, je déplace les calculs sur Colab. Je commence par mesurer le socle et le premier LoRA. J'examine ensuite les réponses et le protocole avant de lancer une version corrigée.

```text
archive vérifiée → comparaison initiale → diagnostic du format
                 → pilote corrigé → phase 1 complète → SFT triage et essai de rappel QA
                 → partitions nettoyées → SFT triage v3 → pilote DPO → comparaison
```

**Organisation**

- Je dépose `ocr14_colab.zip` dans `MyDrive/OCR_PROJET_14`.
- L'exécution documentée utilise une G4 (RTX PRO 6000 Blackwell), en BF16.
- Les données et le socle sont lus depuis le stockage temporaire ; les résultats, adaptateurs et checkpoints sont conservés dans Drive.
- Les lancements d'entraînement tournent en sous-processus : je vérifie leur fin dans les cellules de suivi avant d'exécuter l'évaluation suivante.
- Je libère l'environnement GPU après sauvegarde. La lecture des rapports ne nécessite pas de GPU.

Je conserve les premières sorties locales et Colab comme historique. Les sections suivantes suivent l'ordre des observations ; les résultats de validation restent exploratoires, car ce split sert aussi à sélectionner les checkpoints. [Conditions d'utilisation des ressources Colab](https://research.google.com/colaboratory/faq.html).


## 1. Relier Drive et vérifier le transfert

Les données et l'adaptateur sont copiés dans le stockage temporaire pour la lecture. Les prédictions et rapports sont écrits dans Drive pour survivre à la fin de session.


In [1]:
from google.colab import drive
from pathlib import Path
import hashlib
import json
import zipfile

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/OCR_PROJET_14")
ARCHIVE = DRIVE / "ocr14_colab.zip"
RACINE = Path("/content/OCR_PROJET_14")
assert ARCHIVE.exists(), f"Déposer l'archive ici : {ARCHIVE}"
RACINE.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(ARCHIVE) as archive:
    for nom in archive.namelist():
        destination = (RACINE / nom).resolve()
        assert destination.is_relative_to(RACINE.resolve()), nom
    archive.extractall(RACINE)

manifest = json.loads((RACINE / "manifest_colab.json").read_text())
for nom, attendu in manifest["sha256"].items():
    observe = hashlib.sha256((RACINE / nom).read_bytes()).hexdigest()
    assert observe == attendu, f"Fichier altéré : {nom}"
print(len(manifest["sha256"]), "fichiers vérifiés")


Mounted at /content/drive
41 fichiers vérifiés


## 2. Installer les bibliothèques et choisir la précision

Je conserve le PyTorch fourni par Colab pour son association avec CUDA. Les versions de Transformers, PEFT et des métriques sont fixées à celles du projet. Si Colab demande un redémarrage de session après installation, je le fais puis reprends les cellules depuis le début.


In [2]:
import subprocess
import sys

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==5.22.1", "peft==0.21.2", "accelerate==1.15.0",
    "scikit-learn==1.9.1", "mlflow==3.17.2",
], check=True)


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'transformers==5.22.1', 'peft==0.21.2', 'accelerate==1.15.0', 'scikit-learn==1.9.1', 'mlflow==3.17.2'], returncode=0)

In [4]:
ADAPTATEUR = RACINE / manifest["adaptateur"]
REVISION = manifest["revision_socle"]

assert ADAPTATEUR.is_dir(), f"Adaptateur introuvable : {ADAPTATEUR}"
print(f"Adaptateur : {ADAPTATEUR}")
print(f"Révision du socle : {REVISION}")

Adaptateur : /content/OCR_PROJET_14/models/phase1_qa_lora_kernel38_20261105_093746/adapter
Révision du socle : ea980cb0a6c2ae4b936e82123acc929f1cec04c1


## 3. Conserver le suivi MLflow

La base SQLite travaille sur le disque temporaire. Je la sauvegarde dans Drive après chaque exécution ; les artefacts sont également conservés dans Drive. Une coupure brutale peut laisser le dernier statut MLflow ouvert, mais les lots JSONL déjà écrits restent réutilisables.


In [5]:
import os
import sqlite3
import shutil
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
import mlflow

BASE_LOCALE = Path("/content/ocr14_colab_mlflow.db")
BASE_DRIVE = DRIVE / "suivi" / "mlflow_colab.db"
BASE_DRIVE.parent.mkdir(parents=True, exist_ok=True)
if not BASE_LOCALE.exists() and BASE_DRIVE.exists():
    shutil.copy2(BASE_DRIVE, BASE_LOCALE)
mlflow.set_tracking_uri(f"sqlite:///{BASE_LOCALE}")
client = mlflow.tracking.MlflowClient()
experience = client.get_experiment_by_name("OCR14-Colab-Evaluation")
EXPERIENCE_ID = experience.experiment_id if experience else client.create_experiment(
    "OCR14-Colab-Evaluation", artifact_location=(DRIVE / "suivi/artefacts").as_uri()
)


def sauvegarder_suivi():
    """Copie une image cohérente de SQLite après une exécution."""
    with sqlite3.connect(BASE_LOCALE) as source, sqlite3.connect(BASE_DRIVE) as destination:
        source.backup(destination)


## 4. Produire les réponses et les métriques


In [3]:
import subprocess
import sys

# Je retire une bibliothèque de quantification inutilisée,
# dont la version préinstallée bloque le chargement du LoRA.
subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "torchao"],
    check=True,
)

# Je vérifie dans un nouveau processus, comme le script d'inférence.
subprocess.run(
    [
        sys.executable,
        "-c",
        (
            "from peft.import_utils import is_torchao_available; "
            "assert not is_torchao_available(), 'torchao reste détecté'; "
            "print('PEFT fonctionne sans torchao : vérification réussie.')"
        ),
    ],
    check=True,
)

CompletedProcess(args=['/usr/bin/python3', '-c', "from peft.import_utils import is_torchao_available; assert not is_torchao_available(), 'torchao reste détecté'; print('PEFT fonctionne sans torchao : vérification réussie.')"], returncode=0)

In [ ]:
import json
import os
import re
import subprocess
import sys
import time
from importlib.metadata import version

import mlflow
import torch

# Je vérifie le GPU réellement attribué par Colab.
assert torch.cuda.is_available(), "Sélectionner un environnement GPU dans Colab."

GPU = torch.cuda.get_device_name(0)
assert "6000" in GPU.upper() and "BLACKWELL" in GPU.upper(), (
    f"Cette cellule est configurée pour la G4. GPU attribué : {GPU}"
)
assert torch.cuda.is_bf16_supported(including_emulation=False), (
    "Le BF16 natif n'est pas disponible : vérifier l'environnement PyTorch/CUDA."
)

PRECISION = "bfloat16"
BATCH_SIZE = 32
TOKENS_SORTIE = {"qa": 256, "triage": 192}

# Je sépare les résultats selon le GPU, la précision et la taille des lots.
nom_gpu = re.sub(r"[^a-zA-Z0-9_-]+", "_", GPU)
SORTIES = (
    DRIVE / "evaluations"
    / f"phase1_{nom_gpu}_{PRECISION}_batch{BATCH_SIZE}"
)
SORTIES.mkdir(parents=True, exist_ok=True)

protocole = {
    "gpu": GPU,
    "dtype": PRECISION,
    "versions": {
        nom: version(nom)
        for nom in (
            "torch", "transformers", "peft",
            "accelerate", "scikit-learn", "mlflow",
        )
    },
    "revision": REVISION,
    "empreintes": manifest["sha256"],
    "split": "validation",
    "batch_size": BATCH_SIZE,
    "max_input_tokens": 2048,
    "max_new_tokens": TOKENS_SORTIE,
}

chemin_protocole = SORTIES / "protocole_colab.json"
if chemin_protocole.exists():
    assert json.loads(chemin_protocole.read_text()) == protocole, (
        "Le protocole diffère des résultats existants. "
        "Choisir un nouveau dossier SORTIES pour cette configuration."
    )
else:
    chemin_protocole.write_text(
        json.dumps(protocole, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU : {GPU} | VRAM : {vram:.1f} Gio")
print(f"Précision : {PRECISION} | Taille des lots : {BATCH_SIZE}")
print(f"Résultats : {SORTIES}", flush=True)


def executer_avec_journal(commande, journal):
    """Affiche la progression et conserve les erreurs dans un fichier."""
    with journal.open("a", encoding="utf-8") as log:
        log.write(f"\n--- Nouvelle exécution : {time.strftime('%Y-%m-%d %H:%M:%S')} ---\n")
        log.flush()

        with subprocess.Popen(
            commande,
            cwd=RACINE,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            env={**os.environ, "PYTHONUNBUFFERED": "1"},
        ) as processus:
            for ligne in processus.stdout:
                print(ligne, end="", flush=True)
                log.write(ligne)
                log.flush()

            code_retour = processus.wait()

    if code_retour != 0:
        raise RuntimeError(
            f"Le script s'est arrêté avec le code {code_retour}. "
            f"Consulter les lignes précédentes ou le journal : {journal}"
        )


for nom, adaptateur in (("socle", None), ("phase1", ADAPTATEUR)):
    dossier = SORTIES / nom
    dossier.mkdir(exist_ok=True)

    for phase, tokens in TOKENS_SORTIE.items():
        predictions = dossier / f"{phase}_validation.jsonl"
        journal = dossier / f"{phase}_validation_execution.log"

        commande = [
            sys.executable,
            str(RACINE / "evaluation/inference.py"),
            "--phase", phase,
            "--split", "validation",
            "--resume",
            "--revision", REVISION,
            "--dtype", PRECISION,
            "--batch-size", str(BATCH_SIZE),
            "--max-new-tokens", str(tokens),
            "--output", str(predictions),
        ]
        if adaptateur is not None:
            commande += ["--adapter", str(adaptateur)]

        print(f"\nÉvaluation : {nom} / {phase}", flush=True)
        debut = time.perf_counter()

        try:
            with mlflow.start_run(
                experiment_id=EXPERIENCE_ID,
                run_name=f"{nom}-{phase}-validation-G4-batch{BATCH_SIZE}",
            ):
                mlflow.log_params({
                    "modele": nom,
                    "phase": phase,
                    "split": "validation",
                    "revision": REVISION,
                    "dtype": PRECISION,
                    "gpu": GPU,
                    "batch_size": BATCH_SIZE,
                    "max_new_tokens": tokens,
                })

                # Le script reprend les lots complets déjà sauvegardés.
                executer_avec_journal(commande, journal)
                mlflow.log_metric(
                    "duree_inference_appel_secondes",
                    time.perf_counter() - debut,
                )

                prefixe = "phase1_qa" if phase == "qa" else "phase2_triage"
                source = (
                    RACINE / "data/processed/sft_deux_phases"
                    / f"{prefixe}_validation.jsonl"
                )
                exemples = [
                    json.loads(ligne)
                    for ligne in source.read_text(encoding="utf-8").splitlines()
                    if ligne.strip()
                ]
                lignes = [
                    json.loads(ligne)
                    for ligne in predictions.read_text(encoding="utf-8").splitlines()
                    if ligne.strip()
                ]

                # Je vérifie la couverture et l'ordre avant de calculer les scores.
                assert [x["id"] for x in lignes] == [x["id"] for x in exemples]
                assert len(lignes) == len({x["id"] for x in lignes})

                executer_avec_journal([
                    sys.executable,
                    str(RACINE / "evaluation/evaluer.py"),
                    str(predictions),
                    "--phase", phase,
                    "--split", "validation",
                ], journal)

                rapport = predictions.with_name(
                    predictions.stem + "_metriques.json"
                )
                valeurs = json.loads(rapport.read_text(encoding="utf-8"))

                if phase == "qa":
                    scores = {
                        "accuracy_qcm": valeurs["qcm"]["accuracy"],
                        "f1_lexical": valeurs["reponses_ouvertes"]["f1_lexical_moyen"],
                    }
                else:
                    scores = {
                        "macro_f1_priorite": valeurs["priorite"]["macro_f1"],
                        "accuracy_esi": valeurs["esi"]["accuracy"],
                        "esi_analysable": valeurs["taux_sortie_esi_valide"],
                    }

                mlflow.log_metrics(scores)
                for fichier in (predictions, rapport, chemin_protocole, journal):
                    mlflow.log_artifact(str(fichier))

                duree = (time.perf_counter() - debut) / 60
                print(
                    f"\nTerminé : {nom}/{phase} — "
                    f"{len(lignes)} réponses — {duree:.1f} min",
                    flush=True,
                )
                print(scores, flush=True)

        finally:
            sauvegarder_suivi()

print("\nLes quatre évaluations sont terminées.")

GPU : NVIDIA RTX PRO 6000 Blackwell Server Edition | VRAM : 95.0 Gio
Précision : bfloat16 | Taille des lots : 32
Résultats : /content/drive/MyDrive/OCR_PROJET_14/evaluations/phase1_NVIDIA_RTX_PRO_6000_Blackwell_Server_Edition_bfloat16_batch32

Évaluation : socle / qa
Reprise : 262/262 réponses conservées.
{
  "nombre": 262,
  "qcm": {
    "nombre": 131,
    "accuracy": 0.0,
    "taux_reponse_analysable": 0.0382
  },
  "reponses_ouvertes": {
    "nombre": 131,
    "f1_lexical_moyen": 0.1302
  },
  "phase": "qa",
  "split": "validation",
  "fichier_predictions": "/content/drive/MyDrive/OCR_PROJET_14/evaluations/phase1_NVIDIA_RTX_PRO_6000_Blackwell_Server_Edition_bfloat16_batch32/socle/qa_validation.jsonl",
  "limite": null
}

Terminé : socle/qa — 262 réponses — 0.1 min
{'accuracy_qcm': 0.0, 'f1_lexical': 0.1302}

Évaluation : socle / triage
Reprise : 238/238 réponses conservées.
{
  "nombre": 238,
  "esi": {
    "accuracy": 0.0294,
    "macro_f1": 0.067,
    "rappel_par_classe": {
      

In [ ]:
# Je confronte quelques générations aux réponses attendues.
for phase, prefixe in (("qa", "phase1_qa"), ("triage", "phase2_triage")):
    source = RACINE / "data/processed/sft_deux_phases" / f"{prefixe}_validation.jsonl"
    references = {
        x["id"]: x
        for ligne in source.read_text(encoding="utf-8").splitlines()
        if ligne.strip()
        for x in [json.loads(ligne)]
    }

    fichier = SORTIES / "phase1" / f"{phase}_validation.jsonl"
    sorties = [
        json.loads(ligne)
        for ligne in fichier.read_text(encoding="utf-8").splitlines()
        if ligne.strip()
    ]

    for sortie in sorties[:3]:
        reference = references[sortie["id"]]
        texte = sortie["prediction"]

        print(f"\n--- {phase} | {sortie['id']} ---")
        print("ATTENDU :", reference["messages"][-1]["content"])
        print("GÉNÉRÉ :", texte[:1500])
        if len(texte) > 1500:
            print("FIN :", repr(texte[-300:]))


--- qa | medquad-dcf9177ddfe9ad26fa451a8a54b34ec9ffab3938d2d416b25ac433ebf41af95d ---
ATTENDU : These resources address the diagnosis or management of pyridoxal 5'-phosphate-dependent epilepsy:  - Genetic Testing Registry: Pyridoxal 5'-phosphate-dependent epilepsy  - MedlinePlus Encyclopedia: Lactic acidosis   These resources from MedlinePlus offer information about the diagnosis and management of various health conditions:  - Diagnostic Tests  - Drug Therapy  - Surgery and Rehabilitation  - Genetic Counseling   - Palliative Care
GÉNÉRÉ : What are the signs and symptoms of pyridoxal 5'-phosphate-dependent epilepsy? The Human Phenotype Ontology provides the following list of signs and symptoms for pyridoxal 5'-phosphate-dependent epilepsy. If the information is available, the table below includes how often the symptom is seen in people with this condition. You can use the MedlinePlus Medical Dictionary to look up the definitions for these medical terms. Signs and Symptoms Approximate n

## 5. Lire la comparaison

Les scores portent sur la validation, déjà utilisée pour sélectionner le checkpoint. Le F1 lexical ne valide pas la justesse médicale. Je lis aussi les réponses brutes : la présence d'une lettre correcte peut masquer une continuation répétitive.


In [ ]:
import pandas as pd

lignes = []
for nom in ("socle", "phase1"):
    qa = json.loads((SORTIES / nom / "qa_validation_metriques.json").read_text())
    triage = json.loads((SORTIES / nom / "triage_validation_metriques.json").read_text())
    lignes.append({
        "modèle": nom, "QCM accuracy": qa["qcm"]["accuracy"],
        "QCM analysable": qa["qcm"]["taux_reponse_analysable"],
        "F1 lexical": qa["reponses_ouvertes"]["f1_lexical_moyen"],
        "Priorité macro-F1": triage["priorite"]["macro_f1"],
        "ESI accuracy": triage["esi"]["accuracy"],
        "ESI analysable": triage["taux_sortie_esi_valide"],
    })
tableau = pd.DataFrame(lignes).set_index("modèle")
tableau.to_csv(SORTIES / "comparaison.csv")
display(tableau)
print("Résultats et suivi :", SORTIES, BASE_DRIVE)


,QCM accuracy,QCM analysable,F1 lexical,Priorité macro-F1,ESI accuracy,ESI analysable
modèle,,,,,,
socle,0.0000,0.0382,0.1302,0.0579,0.0294,0.1008
phase1,0.5573,1.0000,0.3247,0.0000,0.0000,0.0000


Résultats et suivi : /content/drive/MyDrive/OCR_PROJET_14/evaluations/phase1_NVIDIA_RTX_PRO_6000_Blackwell_Server_Edition_bfloat16_batch32 /content/drive/MyDrive/OCR_PROJET_14/suivi/mlflow_colab.db


## 6. Diagnostiquer avant de poursuivre

Je distingue le gain mesuré de la qualité réellement observée.

| Validation | Socle | LoRA phase 1 initial | Lecture |
|---|---:|---:|---|
| QCM corrects (131) | 0 % | 55,73 % | Le format de réponse devient exploitable. |
| F1 lexical QA ouvertes (131) | 0,1302 | 0,3247 | Le vocabulaire se rapproche des références ; la justesse reste à examiner. |
| ESI analysable (238) | 10,08 % | 0 % | Aucun triage exploitable après cette phase QA. |

Je relève dans les réponses ouvertes des répétitions, des listes stéréotypées et des éléments absents des références. Le triage n'a pas encore été entraîné : son échec ne suffit pas à invalider la spécialisation QA, mais interdit de revendiquer une capacité de triage.

**Constats techniques vérifiés**

- Le prompt d'inférence ajoute `<think>\n\n</think>` ; le script SFT initial ne l'ajoutait pas.
- Les embeddings de `im_start` et `im_end` sont identiques dans le socle inspecté, avec une tête de sortie liée et gelée. Je ne peux pas compter sur leur distinction comme signal d'arrêt.
- La troncature initiale supprimait la fin de 115 réponses anglaises du train et de 14 de la validation.

Ces défauts sont établis ; leur contribution respective aux scores n'est pas isolée.

```text
v1 conservée → prompt partagé + EOS natif + réponses complètes
             → pilote technique → phase 1 corrigée → réévaluation → phase 2 triage
```

Je repars du socle pour la phase 1 corrigée. Je garde les jeux de données et les réponses de référence inchangés. La longueur maximale observée du train QA est de **4 797 tokens** ; un plafond de **5 120** permet de tout conserver. Un dépassement provoque une erreur explicite.

Les cellules suivantes matérialisent les trois scripts utilisés : format commun, apprentissage, puis inférence instrumentée. Le paramètre `eos_token_id` arrête la génération lorsqu'il est produit ; `max_new_tokens` reste un plafond, pas une preuve d'arrêt appris ([documentation Transformers](https://huggingface.co/docs/transformers/main_classes/text_generation)).

Le pilote utilise 128 exemples du train, 32 de validation et 40 pas, avec une graine fixe. Il teste le pipeline ; ses scores ne remplacent pas ceux de l'entraînement complet. Je n'utilise pas le test dans cette correction ; le pilote historique de dix cas est signalé en section 9.


In [6]:
%%writefile /content/OCR_PROJET_14/training/format_sft.py
"""Je partage le même préfixe entre l'apprentissage SFT et la génération."""

FORMAT_SFT = "template_sans_reflexion_eos_natif_v2"


def construire_prompt(tokenizer, messages):
    """Prépare la réponse sans réflexion, selon le template exact du tokenizer."""
    if tokenizer.chat_template:
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
            enable_thinking=False,
        )
    blocs = [f"<|im_start|>{m['role']}\n{m['content']}<|im_end|>" for m in messages]
    return "\n".join(blocs) + "\n<|im_start|>assistant\n"


def encoder_exemple(tokenizer, exemple, longueur_max):
    """Masque le prompt et apprend la réponse complète, suivie de l'EOS natif.

    Je refuse un exemple trop long : couper sa réponse enseignerait une fin
    artificielle, et couper le cas pourrait supprimer une information utile.
    Le notebook mesure les longueurs avant de fixer le budget de contexte.
    """
    messages = exemple["messages"]
    if [m["role"] for m in messages] != ["system", "user", "assistant"]:
        raise ValueError("Le corpus attend un dialogue system/user/assistant.")
    if tokenizer.eos_token_id is None:
        raise ValueError("Le tokenizer doit définir un token EOS natif.")
    prompt = tokenizer.encode(construire_prompt(tokenizer, messages[:2]), add_special_tokens=False)
    reponse = tokenizer.encode(messages[2]["content"], add_special_tokens=False)
    reponse += [tokenizer.eos_token_id]
    if len(prompt) + len(reponse) > longueur_max:
        raise ValueError(
            f"{exemple.get('id', 'exemple')} : {len(prompt) + len(reponse)} tokens "
            f"> {longueur_max}. Augmenter --max-length ; aucune troncature silencieuse."
        )
    return {"input_ids": prompt + reponse, "labels": [-100] * len(prompt) + reponse,
            "length": len(prompt) + len(reponse)}


Writing /content/OCR_PROJET_14/training/format_sft.py


In [7]:
%%writefile /content/OCR_PROJET_14/training/entrainer_lora.py
"""J'entraîne un adaptateur LoRA SFT et je trace le run dans MLflow."""

import argparse
import hashlib
import json
import os
from pathlib import Path
import random
import sys
import time
from importlib.metadata import version

import mlflow
import torch
from peft import LoraConfig, PeftModel, get_peft_model
from torch.utils.data import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    Trainer,
    TrainerCallback,
    TrainingArguments,
)


RACINE = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(RACINE))
from training.format_sft import FORMAT_SFT, encoder_exemple
DONNEES = RACINE / "data/processed/sft_deux_phases"
MODELE_BASE = "Qwen/Qwen3-1.7B-Base"
SEED = 3407


def lire_jsonl(chemin):
    """Charge les objets non vides d'un fichier JSONL."""
    return [json.loads(ligne) for ligne in chemin.read_text().splitlines() if ligne.strip()]


def empreinte(chemin):
    """Calcule l'empreinte SHA-256 d'un fichier utilisé par le run."""
    return hashlib.sha256(chemin.read_bytes()).hexdigest()


class CorpusSFT(Dataset):
    """Pré-encode un corpus afin de ne pas retokeniser à chaque époque."""

    def __init__(self, tokenizer, exemples, longueur_max):
        self.lignes = [encoder_exemple(tokenizer, x, longueur_max) for x in exemples]

    def __len__(self):
        return len(self.lignes)

    def __getitem__(self, index):
        return self.lignes[index]


class CollateurSFT:
    """Applique un padding dynamique et conserve le masque des labels."""

    def __init__(self, pad_token_id, multiple=8):
        self.pad_token_id = pad_token_id
        self.multiple = multiple

    def __call__(self, exemples):
        longueur = max(len(x["input_ids"]) for x in exemples)
        longueur = ((longueur + self.multiple - 1) // self.multiple) * self.multiple
        ids, labels, masques = [], [], []
        for exemple in exemples:
            marge = longueur - len(exemple["input_ids"])
            ids.append(exemple["input_ids"] + [self.pad_token_id] * marge)
            labels.append(exemple["labels"] + [-100] * marge)
            masques.append([1] * len(exemple["input_ids"]) + [0] * marge)
        return {
            "input_ids": torch.tensor(ids, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
            "attention_mask": torch.tensor(masques, dtype=torch.long),
        }


class JournalMLflow(TrainerCallback):
    """Enregistre dans le run actif les métriques émises par Trainer."""

    def on_log(self, args, state, control, logs=None, **kwargs):
        mesures = {
            cle: float(valeur)
            for cle, valeur in (logs or {}).items()
            if isinstance(valeur, (int, float)) and cle != "epoch"
        }
        if mesures and mlflow.active_run():
            mlflow.log_metrics(mesures, step=int(state.global_step))


def analyser_arguments():
    """Définit les paramètres utiles au smoke test et au run complet."""
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--model", default=MODELE_BASE)
    parser.add_argument("--revision", required=True, help="Révision figée du socle et du tokenizer.")
    parser.add_argument("--dtype", choices=("bfloat16", "float16"), default="bfloat16")
    parser.add_argument("--phase", choices=("phase1_qa", "phase2_triage"), default="phase1_qa")
    parser.add_argument("--adapter", type=Path, help="Adaptateur de phase 1 à poursuivre en phase 2.")
    parser.add_argument("--tracking-uri", default=os.environ.get("MLFLOW_TRACKING_URI", f"sqlite:///{RACINE / 'mlflow.db'}"))
    parser.add_argument("--experiment", default="OCR_PROJET_14-SFT")
    parser.add_argument("--artifact-location", help="Dossier des artefacts pour une nouvelle expérience.")
    parser.add_argument("--train", type=Path, default=DONNEES / "phase1_qa_train.jsonl")
    parser.add_argument("--validation", type=Path, default=DONNEES / "phase1_qa_validation.jsonl")
    parser.add_argument("--output", type=Path, default=RACINE / "models/phase1_qa_lora")
    parser.add_argument("--run-name", default="phase1-qa-lora")
    parser.add_argument("--epochs", type=float, default=3.0)
    parser.add_argument("--learning-rate", type=float, default=2e-4)
    parser.add_argument("--max-length", type=int, default=5120)
    parser.add_argument("--batch-size", type=int, default=1)
    parser.add_argument("--gradient-accumulation", type=int, default=16)
    parser.add_argument("--train-limit", type=int)
    parser.add_argument("--validation-limit", type=int)
    parser.add_argument("--max-steps", type=int, default=-1)
    parser.add_argument("--warmup-steps", type=int, help="Fixe le warmup pour comparer des runs à budget égal.")
    parser.add_argument("--checkpoint-steps", type=int, default=50)
    parser.add_argument("--resume-from-checkpoint", type=Path)
    parser.add_argument("--seed", type=int, default=SEED)
    return parser.parse_args()


def main():
    """Charge les données, entraîne l'adaptateur et sauvegarde le meilleur état."""
    args = analyser_arguments()
    random.seed(args.seed)
    torch.manual_seed(args.seed)
    args.output = args.output.resolve()
    if args.output.exists() and any(args.output.iterdir()) and not args.resume_from_checkpoint:
        raise ValueError("Le dossier de sortie contient déjà un run. Choisir un nouveau dossier.")
    args.output.mkdir(parents=True, exist_ok=True)

    train = lire_jsonl(args.train)
    validation = lire_jsonl(args.validation)
    if args.train_limit:
        train = random.Random(args.seed).sample(train, min(args.train_limit, len(train)))
    if args.validation_limit:
        validation = random.Random(args.seed).sample(validation, min(args.validation_limit, len(validation)))

    tokenizer = AutoTokenizer.from_pretrained(args.model, revision=args.revision)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    corpus_train = CorpusSFT(tokenizer, train, args.max_length)
    corpus_validation = CorpusSFT(tokenizer, validation, args.max_length)

    protocole = {
        "arguments": {k: str(v) if isinstance(v, Path) else v for k, v in vars(args).items()},
        "format_sft": FORMAT_SFT, "eos_token_id": tokenizer.eos_token_id,
        "train_ids": [x["id"] for x in train], "validation_ids": [x["id"] for x in validation],
        "max_tokens_train": max(x["length"] for x in corpus_train.lignes),
        "max_tokens_validation": max(x["length"] for x in corpus_validation.lignes),
        "train_sha256": empreinte(args.train), "validation_sha256": empreinte(args.validation),
        "scripts_sha256": {str(p.relative_to(RACINE)): empreinte(p) for p in (
            Path(__file__), RACINE / "training/format_sft.py")},
        "versions": {nom: version(nom) for nom in ("torch", "transformers", "peft", "mlflow")},
    }
    chemin_protocole = args.output / "protocole_entrainement.json"
    if args.resume_from_checkpoint:
        precedent = json.loads(chemin_protocole.read_text())
        # Seul le chemin de reprise peut changer entre deux morceaux du même run.
        for document in (precedent, protocole):
            document["arguments"].pop("resume_from_checkpoint", None)
        if precedent != protocole:
            raise ValueError("Le protocole du checkpoint diffère ; je refuse une reprise incompatible.")
    else:
        chemin_protocole.write_text(json.dumps(protocole, ensure_ascii=False, indent=2) + "\n")

    modele = AutoModelForCausalLM.from_pretrained(
        args.model, revision=args.revision, dtype=getattr(torch, args.dtype),
    )
    modele.config.use_cache = False
    modele.gradient_checkpointing_enable()
    modele.enable_input_require_grads()
    configuration_lora = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    )
    if args.adapter:
        origine = json.loads((args.adapter.parent / "protocole_entrainement.json").read_text())
        if origine["format_sft"] != FORMAT_SFT or origine["arguments"]["revision"] != args.revision:
            raise ValueError("L'adaptateur initial doit utiliser le même format et le même socle.")
        modele = PeftModel.from_pretrained(modele, str(args.adapter), is_trainable=True)
        configuration_lora = modele.peft_config["default"]
    else:
        modele = get_peft_model(modele, configuration_lora)
    entrainables, total = modele.get_nb_trainable_parameters()

    parametres = TrainingArguments(
        output_dir=str(args.output / "checkpoints"),
        num_train_epochs=args.epochs,
        max_steps=args.max_steps,
        per_device_train_batch_size=args.batch_size,
        per_device_eval_batch_size=args.batch_size,
        gradient_accumulation_steps=args.gradient_accumulation,
        learning_rate=args.learning_rate,
        lr_scheduler_type="cosine",
        warmup_steps=(args.warmup_steps if args.warmup_steps is not None
                      else 20 if args.max_steps < 0 else min(2, args.max_steps)),
        weight_decay=0.01,
        bf16=args.dtype == "bfloat16",
        fp16=args.dtype == "float16",
        tf32=torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8,
        gradient_checkpointing=True,
        use_cache=False,
        eval_strategy="steps",
        eval_steps=(args.checkpoint_steps if args.max_steps < 0
                    else min(args.checkpoint_steps, args.max_steps)),
        save_strategy="steps",
        save_steps=(args.checkpoint_steps if args.max_steps < 0
                    else min(args.checkpoint_steps, args.max_steps)),
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        logging_steps=5 if args.max_steps < 0 else 1,
        logging_first_step=True,
        report_to="none",
        seed=args.seed,
        data_seed=args.seed,
        remove_unused_columns=False,
        optim="adamw_torch_fused",
    )

    mlflow.set_tracking_uri(args.tracking_uri)
    client = mlflow.tracking.MlflowClient()
    if client.get_experiment_by_name(args.experiment) is None:
        client.create_experiment(args.experiment, artifact_location=args.artifact_location)
    mlflow.set_experiment(args.experiment)
    os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
    debut = time.perf_counter()
    with mlflow.start_run(run_name=args.run_name) as run:
        mlflow.log_params({
            "model": args.model,
            "phase": args.phase,
            "revision": args.revision,
            "dtype": args.dtype,
            "format_sft": FORMAT_SFT,
            "initial_adapter": str(args.adapter) if args.adapter else "aucun",
            "train_examples": len(train),
            "validation_examples": len(validation),
            "train_sha256": empreinte(args.train),
            "validation_sha256": empreinte(args.validation),
            "epochs": args.epochs,
            "max_steps": args.max_steps,
            "learning_rate": args.learning_rate,
            "max_length": args.max_length,
            "batch_size": args.batch_size,
            "gradient_accumulation": args.gradient_accumulation,
            "effective_batch_size": args.batch_size * args.gradient_accumulation,
            "seed": args.seed,
            "checkpoint_steps": args.checkpoint_steps,
            "warmup_steps": parametres.warmup_steps,
            "lora_r": configuration_lora.r,
            "lora_alpha": configuration_lora.lora_alpha,
            "lora_dropout": configuration_lora.lora_dropout,
            "trainable_parameters": entrainables,
            "total_parameters": total,
            "output_dir": str(args.output),
        })
        mlflow.log_artifact(str(chemin_protocole), artifact_path="configuration")
        trainer = Trainer(
            model=modele,
            args=parametres,
            train_dataset=corpus_train,
            eval_dataset=corpus_validation,
            data_collator=CollateurSFT(tokenizer.pad_token_id),
            processing_class=tokenizer,
            callbacks=[JournalMLflow()],
        )
        resultat = trainer.train(
            resume_from_checkpoint=(
                str(args.resume_from_checkpoint.resolve())
                if args.resume_from_checkpoint else None
            )
        )
        evaluation = trainer.evaluate()
        modele.save_pretrained(args.output / "adapter")
        tokenizer.save_pretrained(args.output / "adapter")
        trainer.save_state()
        duree = time.perf_counter() - debut
        resume = {
            "mlflow_run_id": run.info.run_id,
            "duree_secondes": round(duree, 2),
            "train_metrics": resultat.metrics,
            "validation_metrics": evaluation,
            "trainable_parameters": entrainables,
            "total_parameters": total,
        }
        (args.output / "resume_entrainement.json").write_text(
            json.dumps(resume, ensure_ascii=False, indent=2) + "\n"
        )
        mlflow.log_metrics({
            "duree_secondes": duree,
            **{f"final_{k}": float(v) for k, v in evaluation.items() if isinstance(v, (int, float))},
        })
        mlflow.log_artifact(str(args.output / "resume_entrainement.json"), artifact_path="resume")
        mlflow.log_artifact(str(args.output / "adapter/adapter_config.json"), artifact_path="configuration")
        print(json.dumps(resume, ensure_ascii=False, indent=2))


if __name__ == "__main__":
    main()


Overwriting /content/OCR_PROJET_14/training/entrainer_lora.py


In [8]:
%%writefile /content/OCR_PROJET_14/evaluation/inference.py
"""Je génère des réponses déterministes pour comparer les checkpoints du projet."""

import argparse
from datetime import datetime, timezone
import json
import os
from pathlib import Path
import sys
import time

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer


RACINE = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(RACINE))
from training.format_sft import construire_prompt
DONNEES = RACINE / "data/processed/sft_deux_phases"
MODELE_BASE = "Qwen/Qwen3-1.7B-Base"


def lire_jsonl(chemin):
    """Charge les objets non vides d'un fichier JSONL."""
    return [json.loads(ligne) for ligne in chemin.read_text().splitlines() if ligne.strip()]


def analyser_arret(ids, fins):
    """Distingue un EOS produit par le modèle d'une coupure à la limite de tokens."""
    for position, identifiant in enumerate(ids):
        if identifiant in fins:
            return {"tokens_sortie": position + 1, "arret": "eos", "token_fin": identifiant}
    return {"tokens_sortie": len(ids), "arret": "limite", "token_fin": None}


def verifier_reprise(predictions, exemples, configuration, taille_lot):
    """Vérifie le préfixe conservé avant d'ajouter des réponses après une interruption."""
    ids = [ligne["id"] for ligne in predictions]
    if len(ids) > len(exemples) or ids != [x["id"] for x in exemples[:len(ids)]] or len(set(ids)) != len(ids):
        raise ValueError("Les identifiants conservés ne correspondent pas au début du corpus.")
    # Les premières sorties locales étaient exclusivement en bfloat16.
    if any(ligne.get(cle, "bfloat16" if cle == "dtype" else None) != valeur
           for ligne in predictions for cle, valeur in configuration.items()):
        raise ValueError("La configuration diffère de celle des réponses conservées.")
    if len(ids) < len(exemples) and len(ids) % taille_lot:
        raise ValueError("Le dernier lot est incomplet ; vérifier le fichier avant de reprendre.")
    return len(ids)


def main():
    """Exécute une inférence batchée et conserve les paramètres avec chaque réponse."""
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--phase", choices=("qa", "triage"), required=True)
    parser.add_argument("--split", choices=("validation", "test"), default="test")
    parser.add_argument("--model", default=MODELE_BASE)
    parser.add_argument("--revision", help="Révision Hugging Face du socle pour figer les poids.")
    parser.add_argument("--adapter", type=Path, help="Adaptateur LoRA à charger sur le modèle de base.")
    parser.add_argument("--output", type=Path, required=True)
    parser.add_argument("--resume", action="store_true", help="Conserve les lots complets déjà produits.")
    parser.add_argument("--dtype", choices=("bfloat16", "float16"), default="bfloat16",
                        help="float16 permet l'inférence sur un GPU sans bfloat16 natif.")
    parser.add_argument("--limit", type=int)
    parser.add_argument("--batch-size", type=int, default=4)
    parser.add_argument("--max-new-tokens", type=int, default=192)
    parser.add_argument("--stop-token", choices=("historique", "eos"), default="historique",
                        help="eos : arrêt natif, pour le SFT corrigé ; historique : EOS ou im_end.")
    args = parser.parse_args()

    prefixe = "phase1_qa" if args.phase == "qa" else "phase2_triage"
    exemples = lire_jsonl(DONNEES / f"{prefixe}_{args.split}.jsonl")
    if args.limit:
        exemples = exemples[:args.limit]
    configuration = {
        "modele": args.model,
        "revision": args.revision,
        "adaptateur": str(args.adapter.resolve()) if args.adapter else None,
        "phase": args.phase,
        "split": args.split,
        "generation": {"do_sample": False, "max_new_tokens": args.max_new_tokens},
        "dtype": args.dtype,
    }
    if args.stop_token != "historique":
        configuration["generation"]["stop_token"] = args.stop_token
    position_depart = 0
    if args.resume and args.output.exists():
        position_depart = verifier_reprise(
            lire_jsonl(args.output), exemples, configuration, args.batch_size
        )
        if args.output.stat().st_size:
            with args.output.open("rb") as fichier:
                fichier.seek(-1, os.SEEK_END)
                if fichier.read(1) != b"\n":
                    raise ValueError("La dernière ligne n'est pas terminée ; vérifier le JSONL.")
        print(f"Reprise : {position_depart}/{len(exemples)} réponses conservées.", flush=True)
        if position_depart == len(exemples):
            return
    tokenizer = AutoTokenizer.from_pretrained(args.model, revision=args.revision)
    tokenizer.padding_side = "left"
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    modele = AutoModelForCausalLM.from_pretrained(
        args.model,
        revision=args.revision,
        dtype=getattr(torch, args.dtype),
        device_map="auto",
    )
    if args.adapter is not None:
        # Je conserve le tokenizer du socle : seul l'adaptateur change dans la comparaison.
        modele = PeftModel.from_pretrained(modele, str(args.adapter.resolve()), is_trainable=False)
    modele.eval()
    fin_message = tokenizer.convert_tokens_to_ids("<|im_end|>")
    fins = [tokenizer.eos_token_id]
    if args.stop_token == "historique" and isinstance(fin_message, int) and fin_message >= 0:
        fins.append(fin_message)
    args.output.parent.mkdir(parents=True, exist_ok=True)
    debut = time.perf_counter()
    with args.output.open("a" if args.resume else "w", encoding="utf-8") as fichier, torch.inference_mode():
        for position in range(position_depart, len(exemples), args.batch_size):
            lot = exemples[position:position + args.batch_size]
            prompts = [construire_prompt(tokenizer, x["messages"][:2]) for x in lot]
            tokens = tokenizer(
                prompts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=2048,
            ).to(modele.device)
            sorties = modele.generate(
                **tokens,
                max_new_tokens=args.max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=fins,
            )
            longueurs = tokens["attention_mask"].sum(dim=1).tolist()
            for exemple, sortie, longueur in zip(lot, sorties, longueurs):
                # Avec le padding gauche, la génération commence après la largeur commune du batch.
                ids_sortie = sortie[tokens["input_ids"].shape[1]:].tolist()
                arret = analyser_arret(ids_sortie, fins)
                prediction = tokenizer.decode(ids_sortie[:arret["tokens_sortie"]], skip_special_tokens=True).strip()
                fichier.write(json.dumps({
                    "id": exemple["id"],
                    "prediction": prediction,
                    "modele": args.model,
                    "revision": args.revision,
                    "adaptateur": str(args.adapter.resolve()) if args.adapter else None,
                    "dtype": args.dtype,
                    "phase": args.phase,
                    "split": args.split,
                    "generation": configuration["generation"],
                    **arret,
                    "date_utc": datetime.now(timezone.utc).isoformat(),
                    "tokens_entree": int(longueur),
                }, ensure_ascii=False) + "\n")
            fichier.flush()
            os.fsync(fichier.fileno())
            print(f"{min(position + len(lot), len(exemples))}/{len(exemples)}", flush=True)
    print(json.dumps({
        "exemples": len(exemples),
        "secondes": round(time.perf_counter() - debut, 2),
        "sortie": str(args.output),
    }, ensure_ascii=False))


if __name__ == "__main__":
    main()


Overwriting /content/OCR_PROJET_14/evaluation/inference.py


In [24]:
# Je vérifie le format et les longueurs sur CPU avant de mobiliser le GPU.
from collections import defaultdict
from transformers import AutoTokenizer
import importlib
import sys

if str(RACINE) not in sys.path:
    sys.path.insert(0, str(RACINE))
import training.format_sft as format_sft
importlib.reload(format_sft)
tokenizer_audit = AutoTokenizer.from_pretrained('Qwen/Qwen3-1.7B-Base', revision=REVISION)
audit_format = {}
for phase in ('phase1_qa', 'phase2_triage'):
    for split in ('train', 'validation'):
        chemin = RACINE / 'data/processed/sft_deux_phases' / f'{phase}_{split}.jsonl'
        exemples = [json.loads(l) for l in chemin.read_text().splitlines()]
        lignes = [format_sft.encoder_exemple(tokenizer_audit, r, 5120) for r in exemples]
        assert all(r['labels'][-1] == tokenizer_audit.eos_token_id for r in lignes)
        audit_format[f'{phase}_{split}'] = {'exemples': len(lignes), 'max_tokens': max(r['length'] for r in lignes)}
print(audit_format)

repartition = defaultdict(lambda: {'exemples': 0, 'tokens_reponse': 0, 'hpo': 0, 'renvoi_ressources': 0})
for ligne in (RACINE / 'data/processed/sft_deux_phases/phase1_qa_train.jsonl').read_text().splitlines():
    r = json.loads(ligne)
    reponse = r['messages'][-1]['content']
    source = 'MedQuAD EN' if r['id'].startswith('medquad-') else 'MediQAl FR'
    repartition[source]['exemples'] += 1
    repartition[source]['tokens_reponse'] += len(tokenizer_audit.encode(reponse, add_special_tokens=False)) + 1
    repartition[source]['hpo'] += int('human phenotype ontology' in reponse.lower())
    repartition[source]['renvoi_ressources'] += int('these resources address' in reponse.lower())
print(json.dumps(repartition, ensure_ascii=False, indent=2))


{'phase1_qa_train': {'exemples': 2098, 'max_tokens': 4797}, 'phase1_qa_validation': {'exemples': 262, 'max_tokens': 2430}, 'phase2_triage_train': {'exemples': 1902, 'max_tokens': 794}, 'phase2_triage_validation': {'exemples': 238, 'max_tokens': 573}}
{
  "MediQAl FR": {
    "exemples": 1049,
    "tokens_reponse": 16308,
    "hpo": 0,
    "renvoi_ressources": 0
  },
  "MedQuAD EN": {
    "exemples": 1049,
    "tokens_reponse": 401593,
    "hpo": 418,
    "renvoi_ressources": 184
  }
}


In [16]:
# Je teste le pipeline corrigé avant de financer l'entraînement complet.
from datetime import datetime, timezone
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

V2 = DRIVE / 'experiences' / 'sft_eos_v2_20261105'
V2.mkdir(parents=True, exist_ok=True)
PILOTE = V2 / 'pilote'
for relatif in ('training/format_sft.py', 'training/entrainer_lora.py', 'evaluation/inference.py'):
    copie = V2 / 'scripts' / relatif
    copie.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(RACINE / relatif, copie)

commande_pilote = [
    sys.executable, str(RACINE / 'training/entrainer_lora.py'),
    '--revision', REVISION, '--dtype', 'bfloat16',
    '--output', str(PILOTE), '--run-name', 'phase1-qa-eos-v2-pilote',
    '--train-limit', '128', '--validation-limit', '32', '--max-steps', '40',
    '--max-length', '5120', '--batch-size', '4', '--gradient-accumulation', '4',
    '--tracking-uri', mlflow.get_tracking_uri(), '--experiment', 'OCR14-Colab-SFT-v2',
    '--artifact-location', (DRIVE / 'suivi/artefacts_sft_v2').as_uri(),
]
# Le calcul tourne en sous-processus ; la cellule suivante contrôle sa fin.
# Je ne relance pas cette cellule pendant qu'un processus est actif.
if 'processus_pilote' in globals() and processus_pilote.poll() is None:
    print('Pilote déjà actif :', processus_pilote.pid)
elif (PILOTE / 'resume_entrainement.json').exists():
    print('Pilote terminé :', PILOTE)
else:
    journal_pilote = V2 / 'pilote_execution.log'
    with journal_pilote.open('a', encoding='utf-8') as journal:
        processus_pilote = subprocess.Popen(
            commande_pilote, cwd=RACINE, stdout=journal, stderr=subprocess.STDOUT,
            env={**os.environ, 'PYTHONUNBUFFERED': '1', 'MLFLOW_DISABLE_AGENT_HINT': '1'},
        )
    print('Pilote lancé, PID :', processus_pilote.pid)
    print('Journal :', journal_pilote)


Pilote lancé, PID : 12652
Journal : /content/drive/MyDrive/OCR_PROJET_14/experiences/sft_eos_v2_20261105/pilote_execution.log


In [18]:
# Je vérifie la progression sans charger le modèle dans le noyau du notebook.
import time
if 'processus_pilote' in globals():
    try:
        processus_pilote.wait(timeout=15)
    except subprocess.TimeoutExpired:
        pass
    print("Code de sortie (None = actif) :", processus_pilote.poll())
print((V2 / 'pilote_execution.log').read_text()[-3500:])
if (PILOTE / 'resume_entrainement.json').exists():
    sauvegarder_suivi()
    print("Résumé :", json.loads((PILOTE / 'resume_entrainement.json').read_text()))


Code de sortie (None = actif) : 0
                                        
{'loss': '0.534', 'grad_norm': '0.3365', 'learning_rate': '2.109e-05', 'epoch': '4.125'}

 85%|████████▌ | 34/40 [00:35<00:05,  1.02it/s]
                                               
{'loss': '0.3855', 'grad_norm': '0.3153', 'learning_rate': '1.628e-05', 'epoch': '4.25'}

 88%|████████▊ | 35/40 [00:36<00:04,  1.13it/s]
                                               
{'loss': '0.9642', 'grad_norm': '0.4801', 'learning_rate': '1.205e-05', 'epoch': '4.375'}

 90%|█████████ | 36/40 [00:37<00:04,  1.02s/it]
                                               
{'loss': '1.176', 'grad_norm': '0.3452', 'learning_rate': '8.423e-06', 'epoch': '4.5'}

 92%|█████████▎| 37/40 [00:39<00:03,  1.20s/it]
                                               
{'loss': '1.041', 'grad_norm': '0.3266', 'learning_rate': '5.418e-06', 'epoch': '4.625'}

 95%|█████████▌| 38/40 [00:39<00:02,  1.07s/it]
                                            

In [19]:
# Je mesure l'arrêt réel sur toute la validation QA, sans toucher au test.
assert (PILOTE / 'resume_entrainement.json').exists(), "Attendre la fin du pilote."
PRED_PILOTE = PILOTE / 'qa_validation.jsonl'
commande = [
    sys.executable, str(RACINE / 'evaluation/inference.py'),
    '--phase', 'qa', '--split', 'validation', '--revision', REVISION,
    '--dtype', 'bfloat16', '--batch-size', '32', '--max-new-tokens', '256',
    '--stop-token', 'eos', '--adapter', str(PILOTE / 'adapter'),
    '--output', str(PRED_PILOTE), '--resume',
]
try:
    with mlflow.start_run(experiment_id=EXPERIENCE_ID, run_name='pilote-eos-v2-validation'):
        with (PILOTE / 'evaluation.log').open('a') as log:
            subprocess.run(commande, check=True, cwd=RACINE, stdout=log, stderr=subprocess.STDOUT)
            subprocess.run([sys.executable, str(RACINE / 'evaluation/evaluer.py'), str(PRED_PILOTE),
                            '--phase', 'qa', '--split', 'validation'],
                           check=True, cwd=RACINE, stdout=log, stderr=subprocess.STDOUT)
        lignes = [json.loads(l) for l in PRED_PILOTE.read_text().splitlines()]
        assert len(lignes) == 262 and len({x['id'] for x in lignes}) == 262
        arrets = {}
        for source in ('mediqal-', 'medquad-'):
            groupe = [r for r in lignes if r['id'].startswith(source)]
            arrets[source] = {'nombre': len(groupe), 'eos': sum(r['arret'] == 'eos' for r in groupe)}
        rapport = json.loads(PRED_PILOTE.with_name('qa_validation_metriques.json').read_text())
        print('Arrêts réels :', arrets)
        print('Métriques :', rapport)
        mlflow.log_params({'format_sft': 'template_sans_reflexion_eos_natif_v2',
                           'etape': 'pilote', 'max_new_tokens': 256, 'batch_size': 32})
        mlflow.log_metrics({'qcm_accuracy': rapport['qcm']['accuracy'],
                           'taux_eos': sum(r['arret'] == 'eos' for r in lignes) / len(lignes)})
        for fichier in (PRED_PILOTE, PRED_PILOTE.with_name('qa_validation_metriques.json')):
            mlflow.log_artifact(str(fichier))
        for source in ('mediqal-', 'medquad-'):
            r = next(r for r in lignes if r['id'].startswith(source))
            print(r['id'], r['arret'], r['prediction'][:1000])
finally:
    sauvegarder_suivi()


Arrêts réels : {'mediqal-': {'nombre': 131, 'eos': 131}, 'medquad-': {'nombre': 131, 'eos': 5}}
Métriques : {'nombre': 262, 'qcm': {'nombre': 131, 'accuracy': 0.4351, 'taux_reponse_analysable': 1.0}, 'reponses_ouvertes': {'nombre': 131, 'f1_lexical_moyen': 0.2788}, 'phase': 'qa', 'split': 'validation', 'fichier_predictions': '/content/drive/MyDrive/OCR_PROJET_14/experiences/sft_eos_v2_20261105/pilote/qa_validation.jsonl', 'limite': None}
mediqal-mcqu-17107 eos Réponse : D) Blocage de l'action des androgènes sur les cellules cibles
medquad-dcf9177ddfe9ad26fa451a8a54b34ec9ffab3938d2d416b25ac433ebf41af95d limite What are the signs and symptoms of pyridoxal 5'-phosphate-dependent epilepsy? The Human Phenotype Ontology provides the following list of signs and symptoms for pyridoxal 5'-phosphate-dependent epilepsy. If the information is available, the table below includes how often the symptom is seen in people with this condition. You can use the MedlinePlus Medical Dictionary to look up th

## 7. Du pilote au run complet

**Observation du pilote.** Les 40 pas se terminent sans erreur. Les 131 QCM de validation produisent l'EOS ; seulement 5/131 réponses ouvertes le produisent avant 256 tokens. L'exemple anglais examiné reste répétitif. Je valide le fonctionnement technique, pas la qualité médicale.

Je repars du socle, sur les **2 098 exemples QA** pendant **3 époques**, avec un batch effectif de 16 et la même configuration LoRA. Le checkpoint est choisi par la loss sur les **262 exemples de validation**. Le pilote n'est pas utilisé comme point de départ.

| Élément | Initial | Corrigé |
|---|---|---|
| Préfixe assistant | Différent entre train et inférence | Même fonction |
| Fin apprise | `im_end`, parfois coupé | EOS natif, toujours conservé |
| Réponses longues | Tronquées | Complètes, plafond 5 120 |
| Suivi | Run initial conservé | Nouveaux runs, scripts et empreintes dans Drive |

Je modifie plusieurs éléments du protocole ensemble : la comparaison mesure leur effet global, pas une ablation causale de chaque correction. Je ne compare pas directement les losses anciennes et nouvelles, car les tokens supervisés diffèrent.

Pour la prochaine comparaison, je régénère **socle, ancien LoRA et nouveau LoRA** avec la même règle d'arrêt native, les mêmes prompts et les mêmes limites (QA : 256 ; triage : 192). Je garde les premières sorties comme historique. Une réponse coupée au plafond peut être longue ou répétitive : je complète le taux d'EOS par une lecture des textes.


In [20]:
# Je repars du socle pour mesurer une phase 1 complète avec le protocole corrigé.
import math

bilan_pilote = json.loads((PILOTE / 'resume_entrainement.json').read_text())
assert math.isfinite(bilan_pilote['validation_metrics']['eval_loss'])
lignes_pilote = [json.loads(l) for l in PRED_PILOTE.read_text().splitlines()]
assert len(lignes_pilote) == 262
# Porte technique minimale : l'EOS doit être effectivement produit dans les deux sources.
# Ce critère ne valide ni la justesse médicale ni la qualité de la phase 1.
for source in ('mediqal-', 'medquad-'):
    assert any(r['arret'] == 'eos' and r['id'].startswith(source) for r in lignes_pilote)
COMPLET = V2 / 'phase1_complete'
commande_complete = [
    sys.executable, str(RACINE / 'training/entrainer_lora.py'),
    '--revision', REVISION, '--dtype', 'bfloat16',
    '--output', str(COMPLET), '--run-name', 'phase1-qa-eos-v2-complete',
    '--epochs', '3', '--max-length', '5120',
    '--batch-size', '4', '--gradient-accumulation', '4', '--checkpoint-steps', '130',
    '--tracking-uri', mlflow.get_tracking_uri(), '--experiment', 'OCR14-Colab-SFT-v2',
    '--artifact-location', (DRIVE / 'suivi/artefacts_sft_v2').as_uri(),
]
if 'processus_complet' in globals() and processus_complet.poll() is None:
    print('Entraînement déjà actif :', processus_complet.pid)
elif (COMPLET / 'resume_entrainement.json').exists():
    print('Entraînement terminé :', COMPLET)
else:
    with (V2 / 'phase1_complete_execution.log').open('a') as journal:
        processus_complet = subprocess.Popen(
            commande_complete, cwd=RACINE, stdout=journal, stderr=subprocess.STDOUT,
            env={**os.environ, 'PYTHONUNBUFFERED': '1', 'MLFLOW_DISABLE_AGENT_HINT': '1'},
        )
    print('Phase 1 complète lancée, PID :', processus_complet.pid)
    print('2 098 exemples, 3 époques, batch effectif 16 ; checkpoints dans Drive.')


Phase 1 complète lancée, PID : 13813
2 098 exemples, 3 époques, batch effectif 16 ; checkpoints dans Drive.


In [27]:
# Je relance cette cellule pour suivre le run ; aucune nouvelle charge GPU n'est créée.
if 'processus_complet' in globals():
    try:
        processus_complet.wait(timeout=15)
    except subprocess.TimeoutExpired:
        pass
    print("Code de sortie (None = actif) :", processus_complet.poll())
journal = (V2 / 'phase1_complete_execution.log').read_text()
print(journal[-1800:])
if (COMPLET / 'resume_entrainement.json').exists():
    sauvegarder_suivi()
    print(json.loads((COMPLET / 'resume_entrainement.json').read_text()))


Code de sortie (None = actif) : 0
[00:00<00:02, 23.76it/s]
100%|██████████| 66/66 [00:04<00:00, 15.53it/s]
{
  "mlflow_run_id": "f1ac257a2fab447f89747fdfdc696448",
  "duree_secondes": 455.87,
  "train_metrics": {
    "train_runtime": 450.3973,
    "train_samples_per_second": 13.974,
    "train_steps_per_second": 0.879,
    "total_flos": 3.3618375556890624e+16,
    "train_loss": 0.7669539048214151,
    "epoch": 3.0
  },
  "validation_metrics": {
    "eval_loss": 0.6601492762565613,
    "eval_runtime": 4.2743,
    "eval_samples_per_second": 61.297,
    "eval_steps_per_second": 15.441,
    "epoch": 3.0
  },
  "trainable_parameters": 17432576,
  "total_parameters": 1738007552
}

{'mlflow_run_id': 'f1ac257a2fab447f89747fdfdc696448', 'duree_secondes': 455.87, 'train_metrics': {'train_runtime': 450.3973, 'train_samples_per_second': 13.974, 'train_steps_per_second': 0.879, 'total_flos': 3.3618375556890624e+16, 'train_loss': 0.7669539048214151, 'epoch': 3.0}, 'validation_metrics': {'eval_loss':

In [12]:
# Je compare les trois états avec une règle d'arrêt strictement identique.
assert (COMPLET / 'resume_entrainement.json').exists(), 'Attendre la fin du run complet.'
EVALUATIONS_V2 = V2 / 'comparaison'
EVALUATIONS_V2.mkdir(exist_ok=True)
protocole_v2 = {
    'revision': REVISION, 'dtype': 'bfloat16', 'batch_size': 32,
    'max_input_tokens': 2048, 'max_new_tokens': {'qa': 256, 'triage': 192},
    'stop_token': 'eos', 'do_sample': False, 'split': 'validation',
    'gpu': torch.cuda.get_device_name(0),
    'empreintes_donnees': {str(p.name): hashlib.sha256(p.read_bytes()).hexdigest()
                          for p in (RACINE / 'data/processed/sft_deux_phases').glob('*_validation.jsonl')},
    'empreintes_scripts': {nom: hashlib.sha256((RACINE / nom).read_bytes()).hexdigest()
                          for nom in ('evaluation/inference.py', 'evaluation/evaluer.py', 'training/format_sft.py')},
}
chemin_v2 = EVALUATIONS_V2 / 'protocole.json'
if chemin_v2.exists():
    assert json.loads(chemin_v2.read_text()) == protocole_v2, 'Protocole différent : choisir un nouveau dossier.'
else:
    chemin_v2.write_text(json.dumps(protocole_v2, ensure_ascii=False, indent=2))

def evaluer_modeles(modeles):
    """Compare les étapes avec le même protocole et journalise chaque tâche."""
    for nom, adaptateur in modeles:
        dossier = EVALUATIONS_V2 / nom
        dossier.mkdir(exist_ok=True)
        for phase, tokens in (('qa', 256), ('triage', 192)):
            predictions = dossier / f'{phase}_validation.jsonl'
            commande = [
                sys.executable, str(RACINE / 'evaluation/inference.py'),
                '--phase', phase, '--split', 'validation', '--resume',
                '--revision', REVISION, '--dtype', 'bfloat16', '--batch-size', '32',
                '--max-new-tokens', str(tokens), '--stop-token', 'eos', '--output', str(predictions),
            ]
            if adaptateur is not None:
                commande += ['--adapter', str(adaptateur)]
            journal = dossier / f'{phase}_execution.log'
            try:
                with mlflow.start_run(experiment_id=EXPERIENCE_ID, run_name=f'{nom}-{phase}-validation-eos-v2'):
                    mlflow.log_params({'modele': nom, 'phase': phase, 'revision': REVISION,
                                       'dtype': 'bfloat16', 'batch_size': 32, 'stop_token': 'eos',
                                       'max_new_tokens': tokens, 'split': 'validation'})
                    with journal.open('a') as log:
                        subprocess.run(commande, check=True, cwd=RACINE, stdout=log, stderr=subprocess.STDOUT)
                        subprocess.run([sys.executable, str(RACINE / 'evaluation/evaluer.py'), str(predictions),
                                        '--phase', phase, '--split', 'validation'],
                                       check=True, cwd=RACINE, stdout=log, stderr=subprocess.STDOUT)
                    lignes = [json.loads(l) for l in predictions.read_text().splitlines()]
                    prefixe = 'phase1_qa' if phase == 'qa' else 'phase2_triage'
                    references = [json.loads(l) for l in (RACINE / 'data/processed/sft_deux_phases' / f'{prefixe}_validation.jsonl').read_text().splitlines()]
                    assert [r['id'] for r in lignes] == [r['id'] for r in references]
                    rapport = predictions.with_name(predictions.stem + '_metriques.json')
                    mesures = json.loads(rapport.read_text())
                    scores = ({'qcm_accuracy': mesures['qcm']['accuracy'],
                               'f1_lexical': mesures['reponses_ouvertes']['f1_lexical_moyen']}
                              if phase == 'qa' else {'macro_f1_priorite': mesures['priorite']['macro_f1'],
                                                    'esi_analysable': mesures['taux_sortie_esi_valide']})
                    scores['taux_eos'] = sum(r['arret'] == 'eos' for r in lignes) / len(lignes)
                    mlflow.log_metrics(scores)
                    for fichier in (predictions, rapport, chemin_v2, journal):
                        mlflow.log_artifact(str(fichier))
                    print(nom, phase, scores, flush=True)
            finally:
                sauvegarder_suivi()

modeles_phase1 = [('socle', None), ('phase1_initiale', ADAPTATEUR), ('phase1_corrigee', COMPLET / 'adapter')]
rapports_phase1 = [EVALUATIONS_V2 / nom / f'{phase}_validation_metriques.json'
                   for nom, _ in modeles_phase1 for phase in ('qa', 'triage')]
if all(p.exists() for p in rapports_phase1):
    print('Six rapports déjà présents : je conserve la comparaison précédente.')
else:
    evaluer_modeles(modeles_phase1)


Six rapports déjà présents : je conserve la comparaison précédente.


## 8. Interpréter les mesures et préparer le triage

Je distingue trois questions :

1. **Le calcul est-il correct ?** Prompt identique, EOS supervisé, corpus intact, checkpoints et MLflow sauvegardés.
2. **Les réponses s'améliorent-elles ?** QCM, F1 lexical, arrêts réels et lecture des mêmes exemples.
3. **Le triage est-il appris ?** Cette capacité relève de la seconde phase ; je mesure aussi l'état intermédiaire pour suivre son évolution.

Le train QA contient 1 049 exemples par langue, mais **16 308 tokens de réponse français contre 401 593 anglais**, EOS compris : l'anglais représente environ **96,1 %** des tokens supervisés. Les tâches diffèrent aussi (QCM français / QA ouvertes anglaises). Je ne peux donc pas attribuer directement une différence de score à la langue.

Dans les références anglaises, je compte 418 mentions de « Human Phenotype Ontology » et 184 de « These resources address ». Ce comptage de chaînes exactes signale des formulations récurrentes ; il ne prouve pas à lui seul une mauvaise qualité. Il motive une revue de l'adéquation des références à des réponses directes.

**Suite prévue :** poursuivre l'adaptateur corrigé sur les 1 902 exemples de triage, sélectionner le checkpoint sur les 238 exemples de validation, puis réévaluer QA et triage pour mesurer le gain métier et un éventuel oubli. Je réserve le test final et le DPO aux étapes suivantes.

**Limite de génération.** Dans la validation anglaise, seules 62 réponses de référence sur 131 tiennent en 256 tokens, EOS compris. Atteindre ce plafond n'établit donc pas, à lui seul, un défaut d'arrêt. Je distingue les réponses réellement répétitives des réponses simplement longues et je conserve ce plafond pour la comparaison historique.


In [30]:
# Je lis les résultats sauvegardés ; cette cellule ne nécessite pas de GPU.
import pandas as pd

lignes_comparaison = []
for nom in ('socle', 'phase1_initiale', 'phase1_corrigee'):
    dossier = EVALUATIONS_V2 / nom
    qa = json.loads((dossier / 'qa_validation_metriques.json').read_text())
    triage = json.loads((dossier / 'triage_validation_metriques.json').read_text())
    sorties_qa = [json.loads(l) for l in (dossier / 'qa_validation.jsonl').read_text().splitlines()]
    resultat = {
        'modèle': nom, 'accuracy_QCM': qa['qcm']['accuracy'],
        'F1_lexical_EN': qa['reponses_ouvertes']['f1_lexical_moyen'],
        'ESI_analysable': triage['taux_sortie_esi_valide'],
        'macro_F1_priorité': triage['priorite']['macro_f1'],
    }
    for langue, prefixe in (('FR', 'mediqal-'), ('EN', 'medquad-')):
        groupe = [r for r in sorties_qa if r['id'].startswith(prefixe)]
        resultat[f'EOS_{langue}'] = sum(r['arret'] == 'eos' for r in groupe) / len(groupe)
    lignes_comparaison.append(resultat)
tableau_v2 = pd.DataFrame(lignes_comparaison).set_index('modèle')
display(tableau_v2)
tableau_v2.to_csv(EVALUATIONS_V2 / 'comparaison_v2.csv')

# Je lis les mêmes exemples pour éviter de choisir seulement des réussites.
source = RACINE / 'data/processed/sft_deux_phases/phase1_qa_validation.jsonl'
references = {r['id']: r for r in map(json.loads, source.read_text().splitlines())}
for prefixe in ('mediqal-', 'medquad-'):
    cibles = [identifiant for identifiant in references if identifiant.startswith(prefixe)][:3]
    for identifiant in cibles:
        print('\nCAS', identifiant)
        print('QUESTION :', references[identifiant]['messages'][1]['content'])
        print('RÉFÉRENCE :', references[identifiant]['messages'][2]['content'][:1200])
        for nom in ('phase1_initiale', 'phase1_corrigee'):
            predictions = [json.loads(l) for l in (EVALUATIONS_V2 / nom / 'qa_validation.jsonl').read_text().splitlines()]
            r = next(r for r in predictions if r['id'] == identifiant)
            print(nom, '| arrêt :', r['arret'], '| tokens :', r['tokens_sortie'])
            print(r['prediction'][:1200])


,accuracy_QCM,F1_lexical_EN,ESI_analysable,macro_F1_priorité,EOS_FR,EOS_EN
modèle,,,,,,
socle,0.0000,0.1302,0.1008,0.0579,0.755725,0.183206
phase1_initiale,0.5573,0.3247,0.0000,0.0000,0.000000,0.000000
phase1_corrigee,0.5954,0.3762,0.0126,0.0065,1.000000,0.244275



CAS mediqal-mcqu-17107
QUESTION : Les anti-androgènes non stéroïdiens agissent par :
A) Inhibition de la sécrétion de testostérone
B) Inhibition de la sécrétion d'hormone lutéotrope (LH) hypophysaire
C) Inhibition de la sécrétion des androgènes surrénaliens
D) Blocage de l'action des androgènes sur les cellules cibles
E) Inhibition de la sécrétion de LH-RH
RÉFÉRENCE : Réponse : D) Blocage de l'action des androgènes sur les cellules cibles
phase1_initiale | arrêt : limite | tokens : 256
Réponse : D) Blocage de l'action des androgènes sur les cellules cibles具有战士ياتينية.公网安
公网安
Réponse : D) Blocage de l'action des androgènes sur les cellules cibles公网安
公网安
Réponse : D) Blocage de l'action des androgènes sur les cellules cibles公网安
公网安
Réponse : D) Blocage de l'action des androgènes sur les cellules cibles公网安
公网安
Réponse : D) Blocage de l'action des androgènes sur les cellules cibles公网安
公网安
Réponse : D) Blocage de l'action des androgènes sur les cellules cibles公网安
公网安
Réponse : D) Blocage d

## 9. Spécialiser le modèle sur le triage

La phase 1 corrigée améliore les QCM (**59,54 % contre 55,73 %**) et le F1 lexical anglais (**0,3762 contre 0,3247**). Les 131 QCM s'arrêtent correctement ; 32/131 réponses anglaises s'arrêtent avant 256 tokens.

La lecture des trois premiers exemples par source confirme une nette amélioration du format des QCM. Les réponses anglaises peuvent encore répéter des listes ou inventer des références. **Je ne considère pas la QA ouverte comme validée médicalement.** Le triage reste analysable dans seulement 3/238 cas : il n'est pas encore appris.

Je poursuis le même adaptateur sur les **1 902 exemples de triage** :

- 3 époques ; batch effectif 16 ; learning rate réduit à `1e-4` pour cette seconde adaptation ;
- contexte de 1 024 tokens : le plus long exemple du train en utilise 794 ;
- sélection du checkpoint par loss sur les 238 exemples de validation ;
- comparaison QA + triage après entraînement, sans réutiliser le test.

Cette étape mesure la spécialisation sur les annotations de consensus. Une réussite sur ces références *silver* ne constitue pas une validation clinique.

**Transparence sur le test :** les corrections présentes utilisent uniquement train et validation. Un pilote historique de dix cas du test triage avait déjà été consulté pour vérifier la chaîne ; cette exposition partielle sera signalée dans l'évaluation finale.


In [29]:
# Je poursuis l'adaptateur corrigé sur le corpus de triage, sans réinitialiser le LoRA.
PHASE2 = V2 / 'phase2_triage'
assert (COMPLET / 'adapter/adapter_model.safetensors').exists()
commande_phase2 = [
    sys.executable, str(RACINE / 'training/entrainer_lora.py'),
    '--revision', REVISION, '--dtype', 'bfloat16', '--phase', 'phase2_triage',
    '--adapter', str(COMPLET / 'adapter'),
    '--train', str(RACINE / 'data/processed/sft_deux_phases/phase2_triage_train.jsonl'),
    '--validation', str(RACINE / 'data/processed/sft_deux_phases/phase2_triage_validation.jsonl'),
    '--output', str(PHASE2), '--run-name', 'phase2-triage-eos-v2',
    '--epochs', '3', '--learning-rate', '1e-4', '--max-length', '1024',
    '--batch-size', '8', '--gradient-accumulation', '2', '--checkpoint-steps', '120',
    '--tracking-uri', mlflow.get_tracking_uri(), '--experiment', 'OCR14-Colab-SFT-v2',
    '--artifact-location', (DRIVE / 'suivi/artefacts_sft_v2').as_uri(),
]
if 'processus_phase2' in globals() and processus_phase2.poll() is None:
    print('Phase 2 déjà active :', processus_phase2.pid)
elif (PHASE2 / 'resume_entrainement.json').exists():
    print('Phase 2 terminée :', PHASE2)
else:
    with (V2 / 'phase2_execution.log').open('a') as journal:
        processus_phase2 = subprocess.Popen(
            commande_phase2, cwd=RACINE, stdout=journal, stderr=subprocess.STDOUT,
            env={**os.environ, 'PYTHONUNBUFFERED': '1', 'MLFLOW_DISABLE_AGENT_HINT': '1'},
        )
    print('Phase 2 lancée, PID :', processus_phase2.pid)


Phase 2 lancée, PID : 19279


In [35]:
# Je contrôle le second entraînement avant son évaluation.
if 'processus_phase2' in globals():
    try:
        processus_phase2.wait(timeout=15)
    except subprocess.TimeoutExpired:
        pass
    print("Code de sortie (None = actif) :", processus_phase2.poll())
print((V2 / 'phase2_execution.log').read_text()[-1800:])
if (PHASE2 / 'resume_entrainement.json').exists():
    sauvegarder_suivi()
    print(json.loads((PHASE2 / 'resume_entrainement.json').read_text()))


Code de sortie (None = actif) : 0
d': '12.61', 'epoch': '3'}

100%|██████████| 357/357 [03:35<00:00,  1.98it/s]

100%|██████████| 30/30 [00:02<00:00, 12.93it/s]

                                               
                                                 
{'train_runtime': '215.9', 'train_samples_per_second': '26.43', 'train_steps_per_second': '1.654', 'train_loss': '0.4102', 'epoch': '3'}

100%|██████████| 357/357 [03:35<00:00,  1.65it/s]

100%|██████████| 30/30 [00:02<00:00, 12.87it/s]
{
  "mlflow_run_id": "b2a7b72c787b44aba9dd0da36c9e7338",
  "duree_secondes": 219.45,
  "train_metrics": {
    "train_runtime": 215.8687,
    "train_samples_per_second": 26.433,
    "train_steps_per_second": 1.654,
    "total_flos": 2.008245607582925e+16,
    "train_loss": 0.4101681537320968,
    "epoch": 3.0
  },
  "validation_metrics": {
    "eval_loss": 0.2720068097114563,
    "eval_runtime": 2.3791,
    "eval_samples_per_second": 100.04,
    "eval_steps_per_second": 12.61,
    "epoch": 3.0
  },


In [36]:
assert (PHASE2 / 'resume_entrainement.json').exists(), 'Attendre la fin de la phase 2.'
evaluer_modeles([('phase2_triage', PHASE2 / 'adapter')])


phase2_triage qa {'qcm_accuracy': 0.0916, 'f1_lexical': 0.0321, 'taux_eos': 1.0}
phase2_triage triage {'macro_f1_priorite': 0.7586, 'esi_analysable': 1.0, 'taux_eos': 1.0}


In [37]:
# Je consolide les résultats pour une lecture sans GPU et une conservation locale.
rapports_finaux = {}
for nom in ('socle', 'phase1_initiale', 'phase1_corrigee', 'phase2_triage'):
    rapports_finaux[nom] = {}
    for phase in ('qa', 'triage'):
        dossier = EVALUATIONS_V2 / nom
        rapport = json.loads((dossier / f'{phase}_validation_metriques.json').read_text())
        predictions = [json.loads(l) for l in (dossier / f'{phase}_validation.jsonl').read_text().splitlines()]
        rapport['arrets_eos'] = sum(r['arret'] == 'eos' for r in predictions)
        rapport['predictions_non_vides'] = sum(bool(r['prediction'].strip()) for r in predictions)
        rapports_finaux[nom][phase] = rapport

entrainements = {}
for nom, dossier in (('pilote', PILOTE), ('phase1_corrigee', COMPLET), ('phase2_triage', PHASE2)):
    resume = json.loads((dossier / 'resume_entrainement.json').read_text())
    etat = json.loads((dossier / 'checkpoints/trainer_state.json').read_text())
    resume['best_model_checkpoint'] = etat.get('best_model_checkpoint')
    resume['global_step'] = etat['global_step']
    resume['eval_loss_history'] = [x for x in etat['log_history'] if 'eval_loss' in x]
    entrainements[nom] = resume

empreintes_adaptateurs = {}
for nom, dossier in (('phase1_initiale', ADAPTATEUR), ('phase1_corrigee', COMPLET / 'adapter'), ('phase2_triage', PHASE2 / 'adapter')):
    with (dossier / 'adapter_model.safetensors').open('rb') as f:
        empreintes_adaptateurs[nom] = hashlib.file_digest(f, 'sha256').hexdigest()

bilan_final = {'rapports': rapports_finaux, 'entrainements': entrainements,
               'empreintes_adaptateurs': empreintes_adaptateurs}
(V2 / 'bilan_final.json').write_text(json.dumps(bilan_final, ensure_ascii=False, indent=2))
print(json.dumps(bilan_final, ensure_ascii=False))

lignes = []
for nom, phases in rapports_finaux.items():
    lignes.append({'modèle': nom, 'QCM': phases['qa']['qcm']['accuracy'],
                   'F1 lexical EN': phases['qa']['reponses_ouvertes']['f1_lexical_moyen'],
                   'ESI exact': phases['triage']['esi']['accuracy'],
                   'Macro-F1 priorité': phases['triage']['priorite']['macro_f1'],
                   'ESI analysable': phases['triage']['taux_sortie_esi_valide'],
                   'Sous-triage critique': phases['triage']['taux_sous_triage_critique']})
tableau_final = pd.DataFrame(lignes).set_index('modèle')
display(tableau_final)
tableau_final.to_csv(V2 / 'comparaison_finale.csv')
sauvegarder_suivi()


{"rapports": {"socle": {"qa": {"nombre": 262, "qcm": {"nombre": 131, "accuracy": 0.0, "taux_reponse_analysable": 0.0382}, "reponses_ouvertes": {"nombre": 131, "f1_lexical_moyen": 0.1302}, "phase": "qa", "split": "validation", "fichier_predictions": "/content/drive/MyDrive/OCR_PROJET_14/experiences/sft_eos_v2_20261105/comparaison/socle/qa_validation.jsonl", "limite": null, "arrets_eos": 123, "predictions_non_vides": 262}, "triage": {"nombre": 238, "esi": {"accuracy": 0.0294, "macro_f1": 0.067, "rappel_par_classe": {"1": 0.125, "2": 0.0, "3": 0.0882, "4": 0.0, "5": 0.0}, "support_par_classe": {"1": 8, "2": 92, "3": 68, "4": 52, "5": 18}, "ic95_rappel": {"1": [0.0224, 0.4709], "2": [-0.0, 0.0401], "3": [0.0411, 0.1794], "4": [0.0, 0.0688], "5": [0.0, 0.1759]}, "matrice_confusion": [[1, 0, 1, 0, 0, 6], [1, 0, 10, 0, 0, 81], [0, 1, 6, 0, 0, 61], [0, 0, 4, 0, 0, 48], [0, 0, 0, 0, 0, 18], [0, 0, 0, 0, 0, 0]], "ordre_matrice": ["1", "2", "3", "4", "5", "invalide"]}, "priorite": {"accuracy": 0.

,QCM,F1 lexical EN,ESI exact,Macro-F1 priorité,ESI analysable,Sous-triage critique
modèle,,,,,,
socle,0.0000,0.1302,0.0294,0.0579,0.1008,0.4583
phase1_initiale,0.5573,0.3247,0.0000,0.0000,0.0000,NaN
phase1_corrigee,0.5954,0.3762,0.0042,0.0065,0.0126,0.0000
phase2_triage,0.0916,0.0321,0.7185,0.7586,1.0000,0.0252


In [38]:
# Je rends les erreurs de triage inspectables, au-delà du score moyen.
from evaluation.metrics import extraire_esi
references_triage = [json.loads(l) for l in (RACINE / 'data/processed/sft_deux_phases/phase2_triage_validation.jsonl').read_text().splitlines()]
predictions_triage = {r['id']: r for r in map(json.loads, (EVALUATIONS_V2 / 'phase2_triage/triage_validation.jsonl').read_text().splitlines())}
erreurs_triage = []
for reference in references_triage:
    r = predictions_triage[reference['id']]
    esi = extraire_esi(r['prediction'])
    if esi != reference['esi']:
        erreurs_triage.append({'id': reference['id'], 'id_paire': reference.get('id_paire'),
                              'langue': reference['langue'], 'attendu': reference['esi'], 'predit': esi,
                              'sous_triage_critique': esi is not None and reference['esi'] <= 2 and esi >= 3,
                              'question': reference['messages'][1]['content'],
                              'reference': reference['messages'][2]['content'],
                              'prediction': r['prediction']})
(V2 / 'erreurs_triage_validation.json').write_text(json.dumps(erreurs_triage, ensure_ascii=False, indent=2))
critiques = [r for r in erreurs_triage if r['sous_triage_critique']]
print('Erreurs ESI :', len(erreurs_triage), '/', len(references_triage))
print('Sous-triages critiques :', len(critiques), '/', sum(r['esi'] <= 2 for r in references_triage), 'entrées de référence ESI 1–2')
print('Cas bilingues distincts concernés :', len({r['id_paire'] for r in critiques}))
for r in (critiques[:3] or erreurs_triage[:3]):
    print(json.dumps(r, ensure_ascii=False))
# Je conserve aussi une paire de sorties pour vérifier le format appris.
for reference in references_triage[:2]:
    r = predictions_triage[reference['id']]
    print('EXEMPLE', reference['id'], '| attendu', reference['esi'], '|', r['prediction'])


Erreurs ESI : 67 / 238
Sous-triages critiques : 6 / 100 entrées de référence ESI 1–2
Cas bilingues distincts concernés : 4
{"id": "mediqal-5969c60d532801e8e3351c2ae713ffae87fd9988040b55f32c32cc7a6d8df97e", "id_paire": "mediqal-5969c60d532801e8e3351c2ae713ffae87fd9988040b55f32c32cc7a6d8df97e", "langue": "fr", "attendu": 2, "predit": 3, "sous_triage_critique": true, "question": "Un homme de 55 ans, présentant une cardiopathie ischémique vient consulter pour des palpitations fréquentes", "reference": "Niveau ESI : 2\nPriorité : urgence maximale\nJustification : Palpitations sur cardiopathie ischémique : risque de trouble du rythme dangereux.\nOrientation : Évaluation médicale très rapide en raison du risque élevé.\nCet avis d'orientation ne remplace pas une évaluation médicale.", "prediction": "Niveau ESI : 3\nPriorité : urgence modérée\nJustification : Palpitations fréquentes : biologie et évaluation spécialisée attendues.\nOrientation : Évaluation médicale avec plusieurs ressources prob

## 10. Bilan observé et décisions suivantes

Les deux entraînements corrigés sont terminés. Je retiens les checkpoints **260** pour la phase 1 et **240** pour la phase 2, sélectionnés par la loss de validation. Les durées enregistrées sont respectivement d'environ **7 min 36 s** et **3 min 39 s**, hors préparation et évaluations.

| Validation | Phase 1 initiale | Phase 1 corrigée | Après phase 2 |
|---|---:|---:|---:|
| Accuracy QCM | 55,73 % | 59,54 % | 9,16 % |
| F1 lexical anglais | 0,3247 | 0,3762 | 0,0321 |
| ESI analysable | 0 % | 1,26 % | 100 % |
| Accuracy ESI | 0 % | 0,42 % | 71,85 % |
| Macro-F1 priorité | 0 | 0,0065 | 0,7586 |

**Ce que j'observe**

- Le défaut d'arrêt des QCM est corrigé. Les réponses ouvertes anglaises restent imparfaites : une meilleure proximité lexicale peut accompagner des listes répétitives ou des références inventées.
- La phase 2 apprend le format et la décision de triage, avec **94 % de rappel de l'urgence maximale**.
- Je relève **6 sous-triages critiques sur 100 entrées ESI 1–2** (2,52 % des 238 entrées au total), représentant **4 cas bilingues distincts**.
- L'ESI est identique entre français et anglais pour **88 paires sur 119** (73,95 %). Les deux versions d'un cas ne sont pas des observations indépendantes.
- Avec la règle initiale, l'avertissement est détecté dans **57,14 %** des sorties françaises et **100 %** des sorties anglaises ; ce comptage initial doit être vérifié avant de conclure à une omission (correction du pluriel en section 13).
- La performance QA chute après le SFT de triage. Le taux de réponse QCM analysable descend à **15,27 %** : cette baisse mélange changement de format et erreurs de réponse ; elle ne démontre pas, seule, une disparition des connaissances.

```text
Correction technique validée       Spécialisation triage mesurée
             ✓                                  ✓
             └──── QA ouverte / rétention / cohérence bilingue à améliorer ────┘
```

**La suite que je retiens**

1. Relire les quatre cas distincts de sous-triage critique et les désaccords bilingues, sans modifier leurs références pour augmenter les scores.
2. Tester un petit rappel de QA pendant la phase 2 pour conserver le comportement généraliste ; comparer ce candidat au run de triage pur conservé ici.
3. Vérifier l'adéquation des paires DPO au comportement attendu (triage, prudence, langue, format) avant de lancer le DPO.
4. Figer le protocole et effectuer ensuite la comparaison finale sur le test, en signalant les dix cas déjà exposés lors du pilote historique.

Ces résultats mesurent l'accord avec un consensus de LLM sur la validation, pas une validation clinique ni une performance sur un test indépendant. L'apport de la phase 1 au triage n'est pas isolé : une comparaison « socle → triage directement » constituerait une ablation complémentaire.

## 11. Sauvegarder et libérer le GPU

Je conserve dans `MyDrive/OCR_PROJET_14/experiences/sft_eos_v2_20261105` les scripts, protocoles, checkpoints, adaptateurs, prédictions, rapports et erreurs. Le suivi MLflow est sauvegardé dans `MyDrive/OCR_PROJET_14/suivi`.

Ce point d'arrêt facultatif libère **la VM entière** après vérification des sauvegardes si `LIBERER_GPU_ICI = True`. Je le laisse à `False` pour poursuivre l'essai de rappel QA ; la dernière cellule assure alors la libération. Les fichiers temporaires et les variables sont perdus ; les fichiers Drive restent disponibles. Je peux terminer la lecture et la rédaction sans GPU. [API publique Colab : runtime.unassign](https://github.com/googlecolab/colabtools/blob/main/google/colab/runtime.py).


In [ ]:
# Point d'arrêt facultatif : je le laisse désactivé pour poursuivre l'essai QA.
LIBERER_GPU_ICI = False
if LIBERER_GPU_ICI:
    from google.colab import runtime
    # Je sauvegarde avant de libérer la VM entière, ce qui arrête l'allocation du GPU.
    for nom_processus in ('processus_pilote', 'processus_complet', 'processus_phase2'):
        processus = globals().get(nom_processus)
        assert processus is None or processus.poll() is not None, f'{nom_processus} est encore actif.'
    sauvegarder_suivi()
    assert BASE_DRIVE.exists()
    assert (COMPLET / 'adapter/adapter_model.safetensors').exists()
    assert (EVALUATIONS_V2 / 'phase1_corrigee/qa_validation_metriques.json').exists()
    assert (PHASE2 / 'adapter/adapter_model.safetensors').exists()
    assert (EVALUATIONS_V2 / 'phase2_triage/triage_validation_metriques.json').exists()
    assert (V2 / 'bilan_final.json').exists()
    print('Adaptateur, rapports et MLflow sauvegardés dans Drive. Libération de la VM.', flush=True)
    # Je termine les écritures distantes avant de supprimer la VM.
    from google.colab import drive
    drive.flush_and_unmount(timeout_ms=300_000)
    runtime.unassign()
else:
    print('Je poursuis vers le rappel QA ; la dernière cellule libérera la VM.')


## 12. Conserver la QA pendant la spécialisation au triage

**Hypothèse.** Un rappel de QA pendant le second SFT peut limiter le changement de comportement observé sur les questions générales.

| Paramètre | Triage seul | Triage + rappel QA |
|---|---:|---:|
| Départ | Phase 1 corrigée | Même adaptateur |
| Train | 1 902 triages | 1 900 triages + 212 QA |
| QA français / anglais | — | 106 / 106 |
| Étapes / batch effectif | 357 / 16 | 357 / 16 |
| Learning rate / warmup | 0,0001 / 20 | Identiques |
| Checkpoints | Tous les 120 pas | Identiques |
| Sélection | Loss de validation triage | Identique |
| Contexte maximum | 1 024 | 5 120, sans troncature |

Le rappel représente **10,04 % des exemples et 27,88 % des tokens de réponse**. Je garde le budget d'optimisation ; le temps de calcul peut augmenter avec la longueur des QA. Le modèle voit moins de triages sur ce budget fixe. Il s'agit d'un essai comparatif, sans recherche systématique d'hyperparamètres.

**Contrôle préalable.** Un même prompt anglais apparaît sous deux identifiants dans train et validation. J'exclus la paire FR/EN du train de cet essai. Je conserve les 238 références pour la comparaison historique et calcule également les métriques sur **236 entrées**, sans la paire contaminée. Ce retrait et le rappel QA changent simultanément : leur effet causal n'est donc pas entièrement isolé. Un autre prompt est commun à validation et test ; la paire correspondante devra être exclue du bilan de test indépendant.

**Lecture prévue avant le résultat.** Je compare la récupération des scores QA au coût éventuel sur le macro-F1 priorité, les sous-triages critiques et l'accord bilingue. Un gain QA ne suffit pas à retenir un candidat qui dégrade le triage. Les références restent un consensus de modèles, sans validation clinique.

Après une libération du runtime, je réexécute uniquement le montage, les dépendances, MLflow et les trois cellules `%%writefile`, puis la préparation ci-dessous. La cellule de comparaison précédente restaure `evaluer_modeles` sans recalculer les six rapports déjà présents.


In [9]:
# Après libération de la VM : exécuter d'abord les cellules de montage,
# dépendances, MLflow et les trois cellules %%writefile, sans relancer les anciens runs.
import torch
import pandas as pd
import sys
import subprocess
V2 = DRIVE / 'experiences/sft_eos_v2_20261105'
COMPLET = V2 / 'phase1_complete'
PHASE2 = V2 / 'phase2_triage'
PILOTE = V2 / 'pilote'
EVALUATIONS_V2 = V2 / 'comparaison'
if str(RACINE) not in sys.path:
    sys.path.insert(0, str(RACINE))
assert torch.cuda.is_available(), 'Reconnecter le runtime GPU avant le calcul.'
assert torch.cuda.is_bf16_supported()
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), 'Gio')
assert (COMPLET / 'resume_entrainement.json').exists()


NVIDIA RTX PRO 6000 Blackwell Server Edition 95.0 Gio


In [10]:
# Je réutilise uniquement des QA du train, sans modifier les partitions sources.
import random
from collections import defaultdict
from transformers import AutoTokenizer

DONNEES = RACINE / 'data/processed/sft_deux_phases'
REPLAY = V2 / 'phase2_triage_replay10'
REPLAY.mkdir(parents=True, exist_ok=True)

def lire_lignes(chemin):
    """Charge les exemples JSONL en conservant leur ordre."""
    return [json.loads(l) for l in chemin.read_text().splitlines() if l.strip()]

qa_train = lire_lignes(DONNEES / 'phase1_qa_train.jsonl')
triage_train = lire_lignes(DONNEES / 'phase2_triage_train.jsonl')
hors_train = [r for phase in ('phase1_qa', 'phase2_triage')
              for split in ('validation', 'test')
              for r in lire_lignes(DONNEES / f'{phase}_{split}.jsonl')]
prompts_reserves = {r['messages'][1]['content'].strip() for r in hors_train}
paires_exclues = {r['id_paire'] for r in triage_train
                   if r['messages'][1]['content'].strip() in prompts_reserves}
exclus = [r['id'] for r in triage_train if r['id_paire'] in paires_exclues]
triage_train = [r for r in triage_train if r['id_paire'] not in paires_exclues]
qa_rappel = []
for langue in ('fr', 'en'):
    candidats = sorted((r for r in qa_train if r['langue'] == langue), key=lambda r: r['id'])
    qa_rappel.extend(random.Random(3407).sample(candidats, 106))
melange = triage_train + qa_rappel
random.Random(3407).shuffle(melange)
assert len(melange) == len({r['id'] for r in melange}) == 2112

# Le contrôle porte sur les identifiants, les paires et les prompts exacts.
# Il ne remplace pas l'audit des quasi-doublons effectué lors de l'ingestion.
assert not {r['id'] for r in melange} & {r['id'] for r in hors_train}
assert not {r['id_paire'] for r in melange if r.get('id_paire')} & {
    r['id_paire'] for r in hors_train if r.get('id_paire')}
assert not {r['messages'][1]['content'].strip() for r in melange} & {
    r['messages'][1]['content'].strip() for r in hors_train}

from training.format_sft import encoder_exemple
tokenizer_audit = AutoTokenizer.from_pretrained('Qwen/Qwen3-1.7B-Base', revision=REVISION)
repartition_replay = defaultdict(lambda: {'exemples': 0, 'tokens_reponse': 0, 'max_tokens': 0})
for r in melange:
    encode = encoder_exemple(tokenizer_audit, r, 5120)
    groupe = f"{r['type_tache']} / {r['langue']}"
    compte = repartition_replay[groupe]
    compte['exemples'] += 1
    compte['tokens_reponse'] += sum(t != -100 for t in encode['labels'])
    compte['max_tokens'] = max(compte['max_tokens'], encode['length'])

CHEMIN_REPLAY = REPLAY / 'train_replay10.jsonl'
texte = ''.join(json.dumps(r, ensure_ascii=False) + '\n' for r in melange)
if CHEMIN_REPLAY.exists():
    assert CHEMIN_REPLAY.read_text() == texte, 'Le mélange existant diffère.'
else:
    CHEMIN_REPLAY.write_text(texte, encoding='utf-8')
audit_replay = {
    'seed': 3407, 'repartition': dict(repartition_replay),
    'exclus_doublons_train': exclus, 'paires_exclues_train': sorted(paires_exclues),
    'qa_ids': [r['id'] for r in qa_rappel],
    'train_sha256': hashlib.sha256(CHEMIN_REPLAY.read_bytes()).hexdigest(),
    'sources_sha256': {p.name: hashlib.sha256(p.read_bytes()).hexdigest()
                       for p in DONNEES.glob('phase[12]_*_*.jsonl')
                       if p.name in {f'{phase}_{split}.jsonl' for phase in ('phase1_qa', 'phase2_triage')
                                     for split in ('train', 'validation', 'test')}},
}
(REPLAY / 'audit_replay.json').write_text(json.dumps(audit_replay, ensure_ascii=False, indent=2))
print(json.dumps(repartition_replay, ensure_ascii=False, indent=2))
print('Mélange :', len(melange), 'exemples ; QA :', f'{len(qa_rappel)/len(melange):.2%}')


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.68k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

{
  "triage / en": {
    "exemples": 950,
    "tokens_reponse": 45651,
    "max_tokens": 568
  },
  "triage / fr": {
    "exemples": 950,
    "tokens_reponse": 65128,
    "max_tokens": 794
  },
  "question_reponse / fr": {
    "exemples": 106,
    "tokens_reponse": 1609,
    "max_tokens": 294
  },
  "question_reponse / en": {
    "exemples": 106,
    "tokens_reponse": 41211,
    "max_tokens": 3098
  }
}
Mélange : 2112 exemples ; QA : 10.04%


In [11]:
# Même point de départ, même batch effectif et même budget d'optimisation.
assert (COMPLET / 'adapter/adapter_model.safetensors').exists()
commande_replay = [
    sys.executable, str(RACINE / 'training/entrainer_lora.py'),
    '--revision', REVISION, '--dtype', 'bfloat16', '--phase', 'phase2_triage',
    '--adapter', str(COMPLET / 'adapter'), '--train', str(CHEMIN_REPLAY),
    '--validation', str(DONNEES / 'phase2_triage_validation.jsonl'),
    '--output', str(REPLAY / 'run'), '--run-name', 'phase2-triage-replay10-eos-v2',
    '--max-steps', '357', '--warmup-steps', '20', '--learning-rate', '1e-4',
    '--max-length', '5120', '--batch-size', '8', '--gradient-accumulation', '2',
    '--checkpoint-steps', '120', '--tracking-uri', mlflow.get_tracking_uri(),
    '--experiment', 'OCR14-Colab-SFT-v2',
    '--artifact-location', (DRIVE / 'suivi/artefacts_sft_v2').as_uri(),
]
for relatif in ('training/entrainer_lora.py', 'training/format_sft.py', 'evaluation/inference.py'):
    destination = REPLAY / 'scripts' / relatif
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(RACINE / relatif, destination)
if 'processus_replay' in globals() and processus_replay.poll() is None:
    print('Rappel QA déjà actif :', processus_replay.pid)
elif (REPLAY / 'run/resume_entrainement.json').exists():
    print('Rappel QA déjà terminé :', REPLAY)
else:
    with (REPLAY / 'execution.log').open('a') as journal:
        processus_replay = subprocess.Popen(
            commande_replay, cwd=RACINE, stdout=journal, stderr=subprocess.STDOUT,
            env={**os.environ, 'PYTHONUNBUFFERED': '1', 'MLFLOW_DISABLE_AGENT_HINT': '1'},
        )
    print('Rappel QA lancé, PID :', processus_replay.pid)


Rappel QA lancé, PID : 2062


In [23]:
# La sortie complète reste dans Drive ; cette cellule ne relance aucun calcul.
import re
code_retour = processus_replay.poll() if 'processus_replay' in globals() else None
print('En cours' if code_retour is None else f'Terminé, code : {code_retour}')
lignes_log = (REPLAY / 'execution.log').read_text().replace('\r', '\n').splitlines()
print('\n'.join([l for l in lignes_log if l.strip() and ('loss' in l or '357 [' in l or 'Error' in l or 'Traceback' in l)][-8:]))
if code_retour not in (None, 0):
    raise RuntimeError('Le run a échoué ; consulter le journal avant de poursuivre.')


Terminé, code : 0
100%|██████████| 357/357 [04:10<00:00,  1.42it/s]
{'eval_loss': '0.2769', 'eval_runtime': '2.38', 'eval_samples_per_second': '100', 'eval_steps_per_second': '12.61', 'epoch': '2.705'}
100%|██████████| 357/357 [04:13<00:00,  1.42it/s]
{'train_runtime': '253.9', 'train_samples_per_second': '22.5', 'train_steps_per_second': '1.406', 'train_loss': '0.4582', 'epoch': '2.705'}
100%|██████████| 357/357 [04:13<00:00,  1.41it/s]
    "train_loss": 0.4581908364088929,
    "eval_loss": 0.27689945697784424,


In [24]:
# Je conserve les mêmes 262 QA et 238 triages pour la comparaison historique.
assert (REPLAY / 'run/resume_entrainement.json').exists(), 'Attendre la fin du run.'
# La fonction evaluer_modeles est définie dans la comparaison de phase 1.
evaluer_modeles([('phase2_replay10', REPLAY / 'run/adapter')])


phase2_replay10 qa {'qcm_accuracy': 0.5573, 'f1_lexical': 0.4201, 'taux_eos': 0.6679389312977099}
phase2_replay10 triage {'macro_f1_priorite': 0.7443, 'esi_analysable': 1.0, 'taux_eos': 1.0}


## 13. Relire les erreurs et vérifier les données du DPO

La relecture des quatre cas distincts sous-triés par le SFT de triage seul montre deux types d'écart au consensus :

| Cas abrégé | Versions en erreur | Observation sur les sorties |
|---|---|---|
| Palpitations et cardiopathie ischémique | FR et EN | La justification évoque les examens attendus, sans reprendre le haut risque de la référence. |
| Vulvite très douloureuse, céphalée et photophobie | FR et EN | Les ressources attendues remplacent là encore la justification de haut risque annotée. |
| Personne âgée cirrhotique, élocution trouble récente | FR | La réponse française est moins urgente que la référence et que la réponse anglaise. |
| Douleur testiculaire et inguinale extrême | FR | Même discordance entre langues ; la réponse produite reste générique. |

Je décris un écart à nos annotations, sans réexpertiser médicalement les cas. Ces exemples de validation servent à l'analyse ; je ne les ajoute pas au train.

**Correction de mesure.** La règle d'avertissement omettait le pluriel « ne remplacent pas », utilisé dans les cibles UltraMedical françaises. Je reconnais désormais le singulier et le pluriel en français et en anglais. Je recalcule le bilan depuis les prédictions conservées et garde le taux initial à titre de traçabilité. Cette présence lexicale ne prouve pas la prudence médicale du contenu.

**Avant le DPO.** L'audit complet des prompts retrouve 30 recoupements entre train DPO et SFT : 26 avec le train QA, 2 avec sa validation et 2 avec son test. Six prompts de validation DPO sont communs au train QA. Je dois traiter ces frontières avant de lancer le DPO.

Sur 4 500 prompts du train DPO, un seul mentionne explicitement ESI ; 7 réponses préférées le mentionnent. Ce contrôle lexical ne suffit pas à classer la pertinence des autres paires, mais les premiers exemples relus sont des QA générales. Les catégories de préférence `hard`, `easy` et `length` ne sont pas des étiquettes de triage. Je ne présente donc pas ce corpus comme une optimisation validée de la sécurité ou de l'ESI. Il faut préciser l'objectif du DPO et auditer ses préférences avant son entraînement.


In [8]:
%%writefile /content/OCR_PROJET_14/evaluation/metrics.py
"""Métriques reproductibles pour la QA médicale et le triage."""

from collections import Counter
import re

from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, recall_score


PRIORITES = ("maximale", "moderee", "differee")
ESI_VERS_PRIORITE = {1: "maximale", 2: "maximale", 3: "moderee", 4: "differee", 5: "differee"}


def extraire_esi(texte):
    """Extrait un niveau ESI explicite compris entre 1 et 5."""
    # Les modèles insèrent parfois du Markdown entre le libellé et la valeur.
    motif = r"(?:niveau\s+)?ESI(?:\s+level|\s+niveau)?[*_\s]*[:=-]?[*_\s]*([1-5])\b"
    trouve = re.search(motif, texte or "", re.IGNORECASE)
    return int(trouve.group(1)) if trouve else None


def extraire_priorite(texte):
    """Normalise les formulations françaises et anglaises des trois priorités."""
    valeur = (texte or "").casefold()
    motifs = {
        "maximale": ("urgence maximale", "maximum urgency", "highest urgency", "immediate urgency"),
        "moderee": ("urgence modérée", "urgence moderee", "moderate urgency"),
        "differee": ("urgence différée", "urgence differee", "delayed urgency", "deferred urgency"),
    }
    for priorite, formulations in motifs.items():
        if any(formulation in valeur for formulation in formulations):
            return priorite
    esi = extraire_esi(texte)
    return ESI_VERS_PRIORITE.get(esi)


def intervalle_wilson(succes, total, z=1.96):
    """Calcule l'intervalle de confiance de Wilson à 95 % d'une proportion."""
    if total == 0:
        return None
    proportion = succes / total
    denominateur = 1 + z * z / total
    centre = proportion + z * z / (2 * total)
    marge = z * ((proportion * (1 - proportion) / total + z * z / (4 * total**2)) ** 0.5)
    return [round((centre - marge) / denominateur, 4), round((centre + marge) / denominateur, 4)]


def _rapport_classes(attendus, predictions, classes):
    """Produit accuracy, macro-F1, rappel et matrice de confusion sur des classes fixes."""
    pred_scoring = [prediction if prediction in classes else "invalide" for prediction in predictions]
    rappel = recall_score(attendus, pred_scoring, labels=list(classes), average=None, zero_division=0)
    supports = Counter(attendus)
    corrects = Counter(a for a, p in zip(attendus, pred_scoring) if a == p)
    return {
        "accuracy": round(accuracy_score(attendus, pred_scoring), 4),
        "macro_f1": round(f1_score(attendus, pred_scoring, labels=list(classes), average="macro", zero_division=0), 4),
        "rappel_par_classe": {classe: round(float(score), 4) for classe, score in zip(classes, rappel)},
        "support_par_classe": {classe: supports[classe] for classe in classes},
        "ic95_rappel": {
            classe: intervalle_wilson(corrects[classe], supports[classe]) for classe in classes
        },
        "matrice_confusion": confusion_matrix(
            attendus, pred_scoring, labels=list(classes) + ["invalide"]
        ).tolist(),
        "ordre_matrice": list(classes) + ["invalide"],
    }


def rapport_triage(exemples, predictions):
    """Évalue l'ESI, la priorité, le sous-triage et le comportement de sortie."""
    par_id = {ligne["id"]: ligne["prediction"] for ligne in predictions}
    lignes = [(exemple, par_id.get(exemple["id"], "")) for exemple in exemples]
    esi_attendus = [x[0]["esi"] for x in lignes]
    esi_predits = [extraire_esi(x[1]) for x in lignes]
    priorites_attendues = [ESI_VERS_PRIORITE[x] for x in esi_attendus]
    priorites_predites = [extraire_priorite(x[1]) for x in lignes]
    valides = [(a, p) for a, p in zip(esi_attendus, esi_predits) if p is not None]
    sous_triages = sum(p > a for a, p in valides)
    sur_triages = sum(p < a for a, p in valides)
    critiques = sum(a <= 2 and p >= 3 for a, p in valides)
    # Les deux corpus français emploient respectivement le singulier et le pluriel.
    # Ce marqueur vérifie la présence d'une formule, pas la prudence clinique du texte.
    disclaimer = ("ne remplace pas", "ne remplacent pas", "does not replace", "do not replace")
    paires = {}
    for (exemple, _), prediction in zip(lignes, esi_predits):
        if exemple.get("id_paire"):
            paires.setdefault(exemple["id_paire"], []).append(prediction)
    paires_completes = [valeurs for valeurs in paires.values() if len(valeurs) == 2]
    paires_valides = [valeurs for valeurs in paires_completes if None not in valeurs]
    rapport = {
        "nombre": len(lignes),
        "esi": _rapport_classes(
            [str(x) for x in esi_attendus],
            [str(x) if x is not None else None for x in esi_predits],
            ("1", "2", "3", "4", "5"),
        ),
        "priorite": _rapport_classes(priorites_attendues, priorites_predites, PRIORITES),
        "taux_sortie_esi_valide": round(len(valides) / len(lignes), 4),
        "erreur_absolue_moyenne_esi": round(sum(abs(a - p) for a, p in valides) / len(valides), 4) if valides else None,
        "taux_sous_triage": round(sous_triages / len(valides), 4) if valides else None,
        "taux_sous_triage_critique": round(critiques / len(valides), 4) if valides else None,
        "taux_sur_triage": round(sur_triages / len(valides), 4) if valides else None,
        "taux_avertissement": round(sum(any(m in texte.casefold() for m in disclaimer) for _, texte in lignes) / len(lignes), 4),
        "coherence_bilingue_esi": {
            "paires": len(paires_completes),
            "paires_analysables": len(paires_valides),
            "taux_meme_prediction": round(
                sum(a == b for a, b in paires_valides) / len(paires_valides), 4
            ) if paires_valides else None,
        },
    }
    rapport["par_langue"] = {
        langue: rapport_triage(
            [exemple for exemple, _ in lignes if exemple["langue"] == langue], predictions
        ) | {"par_langue": None}
        for langue in ("fr", "en")
    } if {x[0]["langue"] for x in lignes} == {"fr", "en"} else None
    return rapport


def extraire_lettre_qcm(texte):
    """Extrait une réponse A-E lorsqu'elle est annoncée explicitement."""
    trouve = re.search(r"(?:réponse|answer|option|choix)\s*[:=-]?\s*([A-E])\b", texte or "", re.I)
    return trouve.group(1).upper() if trouve else None


def _tokens(texte):
    return re.findall(r"\w+", (texte or "").casefold(), re.UNICODE)


def f1_lexical(reference, prediction):
    """Calcule un F1 lexical simple pour les réponses ouvertes."""
    ref, pred = Counter(_tokens(reference)), Counter(_tokens(prediction))
    communs = sum((ref & pred).values())
    if not ref or not pred or communs == 0:
        return 0.0
    precision, rappel = communs / sum(pred.values()), communs / sum(ref.values())
    return 2 * precision * rappel / (precision + rappel)


def rapport_qa(exemples, predictions):
    """Évalue séparément les QCM français et les réponses ouvertes anglaises."""
    par_id = {ligne["id"]: ligne["prediction"] for ligne in predictions}
    qcm, ouvertes = [], []
    for exemple in exemples:
        prediction = par_id.get(exemple["id"], "")
        reference = exemple["messages"][2]["content"]
        lettre = extraire_lettre_qcm(reference)
        if lettre:
            qcm.append((lettre, extraire_lettre_qcm(prediction)))
        else:
            ouvertes.append(f1_lexical(reference, prediction))
    return {
        "nombre": len(exemples),
        "qcm": {
            "nombre": len(qcm),
            "accuracy": round(sum(a == p for a, p in qcm) / len(qcm), 4) if qcm else None,
            "taux_reponse_analysable": round(sum(p is not None for _, p in qcm) / len(qcm), 4) if qcm else None,
        },
        "reponses_ouvertes": {
            "nombre": len(ouvertes),
            "f1_lexical_moyen": round(sum(ouvertes) / len(ouvertes), 4) if ouvertes else None,
        },
    }


Overwriting /content/OCR_PROJET_14/evaluation/metrics.py


In [13]:
# Je contrôle les frontières des partitions sans entraîner ni modifier les données.
from collections import defaultdict, Counter
import re

def prompt_normalise(texte):
    """Repère des doublons de texte malgré la casse et les espaces."""
    return ' '.join(texte.casefold().split())

index_sft = defaultdict(list)
for phase in ('phase1_qa', 'phase2_triage'):
    for split in ('train', 'validation', 'test'):
        for r in lire_lignes(DONNEES / f'{phase}_{split}.jsonl'):
            index_sft[prompt_normalise(r['messages'][1]['content'])].append({
                'phase': phase, 'split': split, 'id': r['id'], 'id_paire': r.get('id_paire')})
doublons_sft = [v for v in index_sft.values() if len({r['split'] for r in v}) > 1]
audit_dpo = {'doublons_sft_entre_partitions': doublons_sft, 'dpo': {}}
for split in ('train', 'validation'):
    fichier = DONNEES / f'dpo_{split}.jsonl'
    if not fichier.exists():
        print(f'{fichier.name} absent de cette archive ; audit disponible localement.')
        continue
    lignes = lire_lignes(fichier)
    collisions = [{'dpo_id': r['id'], 'sft': index_sft[prompt_normalise(r['prompt'][-1]['content'])]}
                  for r in lignes if prompt_normalise(r['prompt'][-1]['content']) in index_sft]
    audit_dpo['dpo'][split] = {
        'nombre': len(lignes), 'collisions': collisions,
        'label_type': dict(Counter(r['label_type'] for r in lignes)),
        # Un marqueur lexical donne un signal de pertinence, pas une qualification clinique.
        'prompts_mentionnant_esi': sum(bool(re.search(r'\bESI\b|emergency severity index',
            r['prompt'][-1]['content'], re.I)) for r in lignes),
        'chosen_mentionnant_esi': sum(bool(re.search(r'\bESI\b|emergency severity index',
            r['chosen'][-1]['content'], re.I)) for r in lignes),
    }
    print(split, len(lignes), 'paires ;', len(collisions), 'prompts communs au SFT')
(V2 / 'audit_partitions_dpo.json').write_text(json.dumps(audit_dpo, ensure_ascii=False, indent=2))
print('Groupes de prompts communs à plusieurs partitions SFT :', len(doublons_sft))


train 4500 paires ; 30 prompts communs au SFT
validation 500 paires ; 6 prompts communs au SFT
Groupes de prompts communs à plusieurs partitions SFT : 2


In [16]:
# Je relis les cas critiques de validation, sans changer leur référence.
erreurs_precedentes = json.loads((V2 / 'erreurs_triage_validation.json').read_text())
print(json.dumps([r for r in erreurs_precedentes if r['sous_triage_critique']], ensure_ascii=False))


[{"id": "mediqal-5969c60d532801e8e3351c2ae713ffae87fd9988040b55f32c32cc7a6d8df97e", "id_paire": "mediqal-5969c60d532801e8e3351c2ae713ffae87fd9988040b55f32c32cc7a6d8df97e", "langue": "fr", "attendu": 2, "predit": 3, "sous_triage_critique": true, "question": "Un homme de 55 ans, présentant une cardiopathie ischémique vient consulter pour des palpitations fréquentes", "reference": "Niveau ESI : 2\nPriorité : urgence maximale\nJustification : Palpitations sur cardiopathie ischémique : risque de trouble du rythme dangereux.\nOrientation : Évaluation médicale très rapide en raison du risque élevé.\nCet avis d'orientation ne remplace pas une évaluation médicale.", "prediction": "Niveau ESI : 3\nPriorité : urgence modérée\nJustification : Palpitations fréquentes : biologie et évaluation spécialisée attendues.\nOrientation : Évaluation médicale avec plusieurs ressources probablement nécessaires.\nCet avis d'orientation ne remplace pas une évaluation médicale."}, {"id": "mediqal-5969c60d532801e8

In [25]:
# Je distingue la comparaison historique et la sensibilité au doublon découvert.
import importlib
import evaluation.metrics as metriques
importlib.reload(metriques)
from evaluation.metrics import rapport_triage, extraire_esi

triage_validation = lire_lignes(DONNEES / 'phase2_triage_validation.jsonl')
train_historique = lire_lignes(DONNEES / 'phase2_triage_train.jsonl')
prompts_train = {prompt_normalise(r['messages'][1]['content']) for r in train_historique}
paires_contaminees = {r['id_paire'] for r in triage_validation
                      if prompt_normalise(r['messages'][1]['content']) in prompts_train}
validation_sans_doublon = [r for r in triage_validation if r['id_paire'] not in paires_contaminees]
assert len(validation_sans_doublon) == 236

bilan_replay = {'rapports': {}, 'paires_validation_exclues_sensibilite': sorted(paires_contaminees)}
bilan_replay['metriques_sha256'] = hashlib.sha256((RACINE / 'evaluation/metrics.py').read_bytes()).hexdigest()
copie_metriques = REPLAY / 'scripts/evaluation/metrics.py'
copie_metriques.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(RACINE / 'evaluation/metrics.py', copie_metriques)
revue_erreurs = {}
for nom in ('phase1_corrigee', 'phase2_triage', 'phase2_replay10'):
    dossier = EVALUATIONS_V2 / nom
    rapports = {}
    for phase in ('qa', 'triage'):
        rapports[phase] = json.loads((dossier / f'{phase}_validation_metriques.json').read_text())
        predictions = lire_lignes(dossier / f'{phase}_validation.jsonl')
        rapports[phase]['arrets_eos'] = sum(r['arret'] == 'eos' for r in predictions)
        if phase == 'triage':
            ancien_avertissement = rapports[phase]['taux_avertissement']
            rapports[phase] = rapport_triage(triage_validation, predictions)
            rapports[phase]['arrets_eos'] = sum(r['arret'] == 'eos' for r in predictions)
            rapports[phase]['taux_avertissement_rapport_initial'] = ancien_avertissement
            rapports['triage_sans_doublon'] = rapport_triage(validation_sans_doublon, predictions)
            par_id = {r['id']: r['prediction'] for r in predictions}
            erreurs = []
            for r in triage_validation:
                texte = par_id[r['id']]
                esi = extraire_esi(texte)
                if esi != r['esi']:
                    erreurs.append({'id': r['id'], 'id_paire': r['id_paire'], 'langue': r['langue'],
                        'esi_reference': r['esi'], 'esi_predit': esi,
                        'critique': esi is not None and r['esi'] <= 2 and esi >= 3,
                        'question': r['messages'][1]['content'],
                        'reference': r['messages'][-1]['content'], 'prediction': texte})
            revue_erreurs[nom] = erreurs
    bilan_replay['rapports'][nom] = rapports

resume_replay = json.loads((REPLAY / 'run/resume_entrainement.json').read_text())
etat_replay = json.loads((REPLAY / 'run/checkpoints/trainer_state.json').read_text())
bilan_replay['entrainement'] = {**resume_replay, 'global_step': etat_replay['global_step'],
    'best_model_checkpoint': etat_replay['best_model_checkpoint'],
    'eval_loss_history': [r for r in etat_replay['log_history'] if 'eval_loss' in r]}
with (REPLAY / 'run/adapter/adapter_model.safetensors').open('rb') as f:
    bilan_replay['adapter_sha256'] = hashlib.file_digest(f, 'sha256').hexdigest()
(V2 / 'bilan_replay.json').write_text(json.dumps(bilan_replay, ensure_ascii=False, indent=2))
(V2 / 'revue_erreurs_replay.json').write_text(json.dumps(revue_erreurs, ensure_ascii=False, indent=2))
print(json.dumps(bilan_replay, ensure_ascii=False))
tableau_replay = pd.DataFrame([{
    'modèle': nom, 'QCM': r['qa']['qcm']['accuracy'],
    'F1 lexical EN': r['qa']['reponses_ouvertes']['f1_lexical_moyen'],
    'ESI exact (238)': r['triage']['esi']['accuracy'],
    'Macro-F1 priorité (238)': r['triage']['priorite']['macro_f1'],
    'Macro-F1 priorité (236)': r['triage_sans_doublon']['priorite']['macro_f1'],
    'Sous-triage critique (238)': r['triage']['taux_sous_triage_critique'],
} for nom, r in bilan_replay['rapports'].items()]).set_index('modèle')
display(tableau_replay)
tableau_replay.to_csv(V2 / 'comparaison_replay.csv')
sauvegarder_suivi()


{"rapports": {"phase1_corrigee": {"qa": {"nombre": 262, "qcm": {"nombre": 131, "accuracy": 0.5954, "taux_reponse_analysable": 1.0}, "reponses_ouvertes": {"nombre": 131, "f1_lexical_moyen": 0.3762}, "phase": "qa", "split": "validation", "fichier_predictions": "/content/drive/MyDrive/OCR_PROJET_14/experiences/sft_eos_v2_20261105/comparaison/phase1_corrigee/qa_validation.jsonl", "limite": null, "arrets_eos": 163}, "triage": {"nombre": 238, "esi": {"accuracy": 0.0042, "macro_f1": 0.0043, "rappel_par_classe": {"1": 0.0, "2": 0.0109, "3": 0.0, "4": 0.0, "5": 0.0}, "support_par_classe": {"1": 8, "2": 92, "3": 68, "4": 52, "5": 18}, "ic95_rappel": {"1": [0.0, 0.3244], "2": [0.0019, 0.059], "3": [0.0, 0.0535], "4": [0.0, 0.0688], "5": [0.0, 0.1759]}, "matrice_confusion": [[0, 0, 0, 0, 0, 8], [0, 1, 0, 0, 0, 91], [1, 0, 0, 0, 0, 67], [0, 1, 0, 0, 0, 51], [0, 0, 0, 0, 0, 18], [0, 0, 0, 0, 0, 0]], "ordre_matrice": ["1", "2", "3", "4", "5", "invalide"]}, "priorite": {"accuracy": 0.0042, "macro_f1":

,QCM,F1 lexical EN,ESI exact (238),Macro-F1 priorité (238),Macro-F1 priorité (236),Sous-triage critique (238)
modèle,,,,,,
phase1_corrigee,0.5954,0.3762,0.0042,0.0065,0.0065,0.0000
phase2_triage,0.0916,0.0321,0.7185,0.7586,0.7551,0.0252
phase2_replay10,0.5573,0.4201,0.7143,0.7443,0.7450,0.0294


In [26]:
# Je compare les mêmes cas avant/après ; ils ne sont pas réinjectés dans le train.
anciens_critiques = {r['id'] for r in revue_erreurs['phase2_triage'] if r['critique']}
nouveaux_critiques = {r['id'] for r in revue_erreurs['phase2_replay10'] if r['critique']}
revue_replay = {
    'critiques_persistants': sorted(anciens_critiques & nouveaux_critiques),
    'critiques_resolus_selon_reference': sorted(anciens_critiques - nouveaux_critiques),
    'nouveaux_critiques': sorted(nouveaux_critiques - anciens_critiques),
    'details_critiques_replay': [r for r in revue_erreurs['phase2_replay10'] if r['critique']],
    'exemples_qa': [],
}
references_qa = lire_lignes(DONNEES / 'phase1_qa_validation.jsonl')
sorties_qa = {nom: {r['id']: r['prediction'] for r in lire_lignes(EVALUATIONS_V2 / nom / 'qa_validation.jsonl')}
              for nom in ('phase1_corrigee', 'phase2_triage', 'phase2_replay10')}
# Les deux premiers exemples de chaque langue : sélection par ordre, pas par réussite.
for langue in ('fr', 'en'):
    for r in [x for x in references_qa if x['langue'] == langue][:2]:
        revue_replay['exemples_qa'].append({'id': r['id'], 'langue': langue,
            'question': r['messages'][1]['content'], 'reference': r['messages'][-1]['content'],
            'predictions': {nom: lignes[r['id']] for nom, lignes in sorties_qa.items()}})
(V2 / 'revue_replay_qualitative.json').write_text(json.dumps(revue_replay, ensure_ascii=False, indent=2))
print(json.dumps(revue_replay, ensure_ascii=False))


{"critiques_persistants": ["mediqal-212e134bd4c608aea485dbdc531167c42366f3c75975d0805244602dab710643", "mediqal-212e134bd4c608aea485dbdc531167c42366f3c75975d0805244602dab710643-en", "mediqal-5969c60d532801e8e3351c2ae713ffae87fd9988040b55f32c32cc7a6d8df97e", "mediqal-5969c60d532801e8e3351c2ae713ffae87fd9988040b55f32c32cc7a6d8df97e-en", "ultramedical-405a576ff747e79d062fb7dd72d15a94d36474ac9aaffddab5495153d54c0633-fr", "ultramedical-efb82076d88d962168a0145e69ed6753005d36595a33c94a999219c6821612b8-fr"], "critiques_resolus_selon_reference": [], "nouveaux_critiques": ["ultramedical-7a32a18b92ccce9a595c450a07cb386be33cec0869b38699373079a473f89607-fr"], "details_critiques_replay": [{"id": "mediqal-5969c60d532801e8e3351c2ae713ffae87fd9988040b55f32c32cc7a6d8df97e", "id_paire": "mediqal-5969c60d532801e8e3351c2ae713ffae87fd9988040b55f32c32cc7a6d8df97e", "langue": "fr", "esi_reference": 2, "esi_predit": 3, "critique": true, "question": "Un homme de 55 ans, présentant une cardiopathie ischémique vi

## 14. Résultat du rappel QA : un compromis, pas un remplacement automatique

Le run est terminé : **357 étapes, 4 min 18 s**, checkpoint **357** retenu par la loss de validation. L'adaptateur, les protocoles, les prédictions et MLflow sont sauvegardés dans Drive.

| Validation | Triage seul | Triage + rappel QA |
|---|---:|---:|
| Accuracy QCM (131) | 9,16 % | **55,73 %** |
| F1 lexical anglais (131) | 0,0321 | **0,4201** |
| ESI exact (238) | **71,85 %** | 71,43 % |
| Macro-F1 priorité (238) | **0,7586** | 0,7443 |
| Macro-F1 priorité, sans doublon (236) | **0,7551** | 0,7450 |
| Sous-triages critiques / 100 entrées ESI 1–2 | **6** | 7 |
| Accord ESI entre langues | 88 / 119 | **89 / 119** |
| ESI analysable | 100 % | 100 % |
| Avertissement détecté, règle corrigée | 100 % | 100 % |

**Ce que j'en retiens**

- Le rappel restaure le comportement QA : sur les deux QCM affichés, le modèle répond à nouveau à la question au lieu de produire un triage. La phase 1 corrigée seule restait à 59,54 % en QCM ; la récupération est donc partielle.
- Les six sous-triages critiques antérieurs persistent. Une nouvelle erreur apparaît sur la version française d'un cas évoquant notamment des crachats de sang. Il reste cinq cas distincts concernés, contre quatre auparavant. Je conserve les références, sans réinjecter ces cas de validation dans le train.
- La hausse du F1 lexical anglais n'est pas une preuve de justesse : la sortie sur Kallmann conserve une liste répétitive de références non étayées par la cible. La qualité des QA sources et celle des générations restent à travailler.
- Le retrait du doublon de validation ne renverse pas le classement sur la métrique principale. Les scores sont exploratoires : une seed, sélection sur validation et observations bilingues corrélées. Je ne revendique pas de différence statistiquement démontrée.

```text
Rappel QA → format généraliste récupéré
          → léger recul sur la priorité et un sous-triage critique supplémentaire
          → candidat conservé pour comparaison, pas promu automatiquement
```

**Décision.** Je garde le SFT de triage seul comme référence pour la priorité métier et le candidat avec rappel pour étudier la rétention QA. Aucun des deux n'est validé cliniquement. Les ratios de sous-triage ne sont interprétables qu'avec le taux de sorties analysables ; ceux de la phase 1, presque toujours non analysable en ESI, ne mesurent pas sa sécurité.

**Prochaine étape.** Je dois assainir les frontières des données DPO et préciser l'objectif des préférences avant d'entraîner : amélioration de la QA générale ou comportements de triage (priorité, prudence, langue et format). Un DPO généraliste n'est pas présumé corriger les erreurs ESI. Je garde les cas de validation hors apprentissage et je définis le périmètre du test indépendant après exclusion des doublons et prise en compte du pilote historique.

Le fichier `bilan_replay.json` conserve les mesures et l'empreinte de l'adaptateur ; `revue_replay_qualitative.json` conserve les exemples comparés. La cellule suivante vérifie les sauvegardes et libère la VM. La rédaction et la lecture du bilan ne nécessitent plus de GPU.


In [ ]:
LIBERER_GPU_APRES_REPLAY = False  # True si je m'arrête à cette étape.
if LIBERER_GPU_APRES_REPLAY:
    # Je libère la VM dès que le calcul et les sauvegardes sont vérifiés.
    from google.colab import runtime
    assert 'processus_replay' not in globals() or processus_replay.poll() == 0
    sauvegarder_suivi()
    for chemin in (BASE_DRIVE, REPLAY / 'run/adapter/adapter_model.safetensors',
                   V2 / 'bilan_replay.json', V2 / 'revue_erreurs_replay.json',
                   EVALUATIONS_V2 / 'phase2_replay10/qa_validation_metriques.json',
                   EVALUATIONS_V2 / 'phase2_replay10/triage_validation_metriques.json'):
        assert chemin.exists() and chemin.stat().st_size > 0, chemin
    print('Calcul et sauvegardes terminés. Libération de la VM.', flush=True)
    # Je termine les écritures distantes avant de supprimer la VM.
    from google.colab import drive
    drive.flush_and_unmount(timeout_ms=300_000)
    runtime.unassign()


## 15. Figer les partitions avant le dernier SFT et le pilote DPO

Je conserve les fichiers historiques et je produis `data/processed/partitions_v3`. Les exclusions sont justifiées dans un manifeste ; aucun cas ne change de partition.

| Partition | Avant | Version v3 | Motif |
|---|---:|---:|---|
| QA train / validation / test | 2 098 / 262 / 262 | identique | Aucun doublon détecté par ce contrôle |
| Triage train | 1 902 | 1 888 | 7 paires retirées : 1 commune à validation, 6 doublons internes |
| Triage validation | 238 | 238 | Références conservées |
| Triage test indépendant | 238 | 226 | 5 paires exposées au pilote et 1 commune à validation |
| DPO train / validation | 4 500 / 500 | 4 470 / 494 | Prompts communs au SFT historique exclus |

Je normalise la casse et les espaces, puis je traite les traductions comme un groupe indivisible. Ce contrôle repère des doublons textuels ; il ne garantit pas l'absence de paraphrases proches. Le corpus SFT exploitable compte maintenant **4 974 entrées** : je privilégie l'intégrité des partitions plutôt qu'un remplissage artificiel jusqu'à 5 000.

```text
Données historiques → exclusions traçables → version v3 figée
Phase 1 corrigée    → SFT triage nettoyé    → pilote DPO → mêmes évaluations QA + triage
```

Après une nouvelle connexion GPU, je réexécute le montage, les dépendances et MLflow, puis les cellules qui écrivent `format_sft.py`, `entrainer_lora.py` et `metrics.py`. Les cellules suivantes reconstruisent les données et les nouveaux scripts sans relancer les expériences historiques.


In [9]:
# Je reprends une VM neuve après les cellules de montage, dépendances et MLflow.
import torch
import pandas as pd
import sys
import subprocess
import shutil
import os
import hashlib
import json
from pathlib import Path
V2 = DRIVE / 'experiences/sft_eos_v2_20261105'
V3 = DRIVE / 'experiences/sft_dpo_v3_20261105'
V3.mkdir(parents=True, exist_ok=True)
COMPLET = V2 / 'phase1_complete'
SFT_V3 = V3 / 'sft_triage'
DPO_V3 = V3 / 'dpo_pilote'
DATA_V3 = RACINE / 'data/processed/partitions_v3'
PREF_V3 = RACINE / 'data/processed/dpo_pilote_v3'
EVALUATIONS_V3 = V3 / 'comparaison'
(RACINE / 'data_ingestion').mkdir(exist_ok=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'trl==1.17.1', 'datasets==4.7.0'], check=True)
if str(RACINE) not in sys.path:
    sys.path.insert(0, str(RACINE))
assert (COMPLET / 'adapter/adapter_model.safetensors').exists()
print('Point de départ : phase 1 corrigée ; versions v3 distinctes des essais historiques.')


Point de départ : phase 1 corrigée ; versions v3 distinctes des essais historiques.


In [10]:
%%writefile /content/OCR_PROJET_14/data_ingestion/figer_partitions.py
"""Je fige une version nettoyée sans redistribuer les partitions historiques."""

import argparse
from collections import defaultdict
import hashlib
import json
from pathlib import Path
import sys

RACINE = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(RACINE))


def lire_jsonl(chemin):
    """Charge les objets JSONL non vides dans l'ordre du fichier."""
    return [json.loads(l) for l in chemin.read_text().splitlines() if l.strip()]


def normaliser(texte):
    """Compare les prompts indépendamment de la casse et des espaces."""
    return ' '.join(texte.casefold().split())


def groupes(lignes):
    """Garde les deux langues d'un cas dans un groupe indivisible."""
    resultat = defaultdict(list)
    for r in lignes:
        resultat[r.get('id_paire', r['id'])].append(r)
    return resultat


def nettoyer(sources, ids_exposes):
    """Privilégie validation, puis test, puis train pour les prompts dupliqués.

    Je retire les groupes entiers en conflit. Le test exposé reste traçable dans
    les exclusions ; il ne devient jamais une nouvelle donnée d'apprentissage.
    Les prompts DPO sont disjoints de tout le SFT historique, même écarté.
    """
    sorties, exclusions, vus = {}, [], set()
    reserves = {normaliser(r['messages'][1]['content']) for nom, lignes in sources.items()
                if nom.startswith('phase') and not nom.endswith('_train') for r in lignes}
    for split in ('validation', 'test', 'train'):
        for phase in ('phase1_qa', 'phase2_triage'):
            nom = f'{phase}_{split}'
            sorties[nom] = []
            for identifiant, lignes in groupes(sources[nom]).items():
                prompts = {normaliser(r['messages'][1]['content']) for r in lignes}
                motif = ('pilote_test_deja_expose' if split == 'test' and
                         any(r['id'] in ids_exposes for r in lignes) else
                         'prompt_commun_autre_groupe' if prompts & (vus | reserves if split == 'train' else vus)
                         else None)
                if motif:
                    exclusions.append({'fichier': nom, 'groupe': identifiant,
                                       'ids': [r['id'] for r in lignes], 'motif': motif})
                else:
                    sorties[nom].extend(lignes)
                    vus.update(prompts)

    prompts_sft = {normaliser(r['messages'][1]['content']) for nom, lignes in sources.items()
                   if nom.startswith('phase') for r in lignes}
    prompts_dpo = set()
    for split in ('validation', 'train'):
        nom = f'dpo_{split}'
        sorties[nom] = []
        for r in sources[nom]:
            prompt = normaliser(r['prompt'][-1]['content'])
            choisi, rejete = (normaliser(r[k][-1]['content']) for k in ('chosen', 'rejected'))
            motif = ('reponse_vide_ou_identique' if not choisi or not rejete or choisi == rejete else
                     'prompt_commun_sft' if prompt in prompts_sft else
                     'prompt_dpo_deja_present' if prompt in prompts_dpo else None)
            if motif:
                exclusions.append({'fichier': nom, 'ids': [r['id']], 'motif': motif})
            else:
                sorties[nom].append(r)
                prompts_dpo.add(prompt)
    return sorties, exclusions


def verifier(sorties):
    """Vérifie identifiants, prompts et intégrité des paires après filtrage."""
    ids_vus, prompts_vus = set(), set()
    for nom, lignes in sorties.items():
        assert len(lignes) == len({r['id'] for r in lignes}), nom
        assert not ids_vus & {r['id'] for r in lignes}, nom
        ids_vus.update(r['id'] for r in lignes)
        prompts = {normaliser(r['messages'][1]['content'] if nom.startswith('phase')
                              else r['prompt'][-1]['content']) for r in lignes}
        assert not prompts_vus & prompts, nom
        prompts_vus.update(prompts)
        if nom.startswith('phase2'):
            assert all(len(g) == 2 and {r['langue'] for r in g} == {'fr', 'en'}
                       for g in groupes(lignes).values()), nom


def ecrire_immuable(chemin, texte):
    """Réexécute à l'identique, mais refuse de modifier une version déjà figée."""
    if chemin.exists() and chemin.read_text() != texte:
        raise ValueError(f'Version existante différente : {chemin}')
    chemin.write_text(texte, encoding='utf-8')


def main():
    """Produit les partitions v3, les exclusions et leurs empreintes."""
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--source', type=Path, default=RACINE / 'data/processed/sft_deux_phases')
    parser.add_argument('--output', type=Path, default=RACINE / 'data/processed/partitions_v3')
    parser.add_argument('--exposition', type=Path, default=RACINE / 'evaluation/pilote_expose.json')
    args = parser.parse_args()
    noms = [f'{p}_{s}' for p in ('phase1_qa', 'phase2_triage') for s in ('train', 'validation', 'test')]
    noms += ['dpo_train', 'dpo_validation']
    sources = {n: lire_jsonl(args.source / f'{n}.jsonl') for n in noms}
    exposition = json.loads(args.exposition.read_text())
    sorties, exclusions = nettoyer(sources, set(exposition['ids']))
    verifier(sorties)
    args.output.mkdir(parents=True, exist_ok=True)
    rapport = {'politique': 'validation > test > train ; exclusion des paires exposées ; DPO disjoint du SFT',
               'exposition_pilote': exposition, 'exclusions': exclusions, 'fichiers': {}}
    for nom, lignes in sorties.items():
        chemin = args.output / f'{nom}.jsonl'
        ecrire_immuable(chemin, ''.join(json.dumps(r, ensure_ascii=False) + '\n' for r in lignes))
        rapport['fichiers'][nom] = {'avant': len(sources[nom]), 'apres': len(lignes),
            'source_sha256': hashlib.sha256((args.source / f'{nom}.jsonl').read_bytes()).hexdigest(),
            'sha256': hashlib.sha256(chemin.read_bytes()).hexdigest()}
    ecrire_immuable(args.output / 'manifest.json', json.dumps(rapport, ensure_ascii=False, indent=2) + '\n')
    print(json.dumps(rapport['fichiers'], ensure_ascii=False, indent=2))


if __name__ == '__main__':
    main()


Writing /content/OCR_PROJET_14/data_ingestion/figer_partitions.py


In [11]:
%%writefile /content/OCR_PROJET_14/evaluation/pilote_expose.json
{
  "source": "data/evaluation/baseline_qwen3_1_7b_base/triage_test_pilote.jsonl",
  "sha256": "247b0b666292bfb1b38efe0c106cce59146fe1759b0d8f61661be8287ecbd8bd",
  "ids": [
    "mediqal-d683ea08c1a6e7bb01f2db8aca19e141a904136005539240ce002190b73f7065",
    "mediqal-d683ea08c1a6e7bb01f2db8aca19e141a904136005539240ce002190b73f7065-en",
    "mediqal-6157994689997802b73009b6f0261dd32c776c3564a99db5ef0be1e74ce31361",
    "mediqal-6157994689997802b73009b6f0261dd32c776c3564a99db5ef0be1e74ce31361-en",
    "mediqal-5a8a712d2bd2e05bd7f9268290327016fe1ad2bcfa7e2f1eaa40193657335221",
    "mediqal-5a8a712d2bd2e05bd7f9268290327016fe1ad2bcfa7e2f1eaa40193657335221-en",
    "mediqal-30775d38634da1ef32e3476dd6767a6999e78f98e5dd530a4c8355dab67239bb",
    "mediqal-30775d38634da1ef32e3476dd6767a6999e78f98e5dd530a4c8355dab67239bb-en",
    "mediqal-6d6f5aefc2c0f590547fb09d873ad2052e06d820103cf2d6b1abf08098603a9f",
    "mediqal-6d6f5aefc2c0f590547fb09d873ad2052e06d820103cf2d6b1abf08098603a9f-en"
  ]
}


Writing /content/OCR_PROJET_14/evaluation/pilote_expose.json


In [12]:
%%writefile /content/OCR_PROJET_14/training/preparer_dpo.py
"""Je prépare un pilote DPO avec le prompt et l'EOS déjà validés en SFT."""

import argparse
from collections import Counter
import hashlib
import json
from pathlib import Path
import sys

RACINE = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(RACINE))
from data_ingestion.figer_partitions import ecrire_immuable, lire_jsonl
from training.format_sft import construire_prompt


def formater_paire(exemple, tokenizer, longueur_max, systeme):
    """Conserve les deux réponses entières et refuse une frontière ambiguë.

    Le format texte évite que TRL réapplique un autre chat template. Les réponses
    finissent par l'EOS natif, comme le SFT, et non par la balise im_end.
    """
    if [m['role'] for m in exemple['prompt']] != ['user']:
        raise ValueError('Le pilote attend une question utilisateur unique.')
    prompt = construire_prompt(tokenizer, [{'role': 'system', 'content': systeme}] + exemple['prompt'])
    ids_prompt = tokenizer.encode(prompt, add_special_tokens=False)
    resultat = {'id': exemple['id'], 'prompt': prompt, 'label_type': exemple['label_type']}
    maximum = 0
    for cle in ('chosen', 'rejected'):
        if [m['role'] for m in exemple[cle]] != ['assistant']:
            raise ValueError('Une seule réponse assistant est attendue.')
        texte = exemple[cle][0]['content']
        if not texte.endswith(tokenizer.eos_token):
            texte += tokenizer.eos_token
        ids = tokenizer.encode(prompt + texte, add_special_tokens=False)
        if ids[:len(ids_prompt)] != ids_prompt or ids[-1] != tokenizer.eos_token_id:
            raise ValueError(f"Frontière de tokenisation incorrecte : {exemple['id']}")
        maximum = max(maximum, len(ids))
        resultat[cle] = texte
    if maximum > longueur_max:
        return None, maximum
    return resultat, maximum


def selectionner(lignes, nombre):
    """Échantillonne par catégorie de préférence, sans utiliser les résultats du modèle."""
    volumes = Counter(r['label_type'] for r in lignes)
    ideaux = {c: min(nombre, len(lignes)) * n / len(lignes) for c, n in volumes.items()}
    quotas = {c: int(n) for c, n in ideaux.items()}
    for c in sorted(quotas, key=lambda c: (-(ideaux[c] - quotas[c]), c))[:min(nombre, len(lignes)) - sum(quotas.values())]:
        quotas[c] += 1
    def ordre_stable(identifiant):
        return hashlib.sha256(f'3407:{identifiant}'.encode()).hexdigest()
    selection = []
    for categorie, quota in sorted(quotas.items()):
        candidats = sorted((r for r in lignes if r['label_type'] == categorie), key=lambda r: ordre_stable(r['id']))
        selection.extend(candidats[:quota])
    return sorted(selection, key=lambda r: ordre_stable(r['id']))


def main():
    """Audite les longueurs puis fige les 256/64 paires du pilote."""
    from transformers import AutoTokenizer
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--data-dir', type=Path, default=RACINE / 'data/processed/partitions_v3')
    parser.add_argument('--output', type=Path, default=RACINE / 'data/processed/dpo_pilote_v3')
    parser.add_argument('--revision', required=True)
    parser.add_argument('--max-length', type=int, default=4096)
    args = parser.parse_args()
    tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen3-1.7B-Base', revision=args.revision)
    systemes = {r['messages'][0]['content'] for r in lire_jsonl(args.data_dir / 'phase1_qa_train.jsonl')
                if r['langue'] == 'en'}
    assert len(systemes) == 1, 'La consigne anglaise SFT doit être unique.'
    systeme = systemes.pop()
    args.output.mkdir(parents=True, exist_ok=True)
    rapport = {'revision': args.revision, 'max_length': args.max_length, 'seed': 3407, 'splits': {}}
    for split, quota in (('train', 256), ('validation', 64)):
        source = args.data_dir / f'dpo_{split}.jsonl'
        eligibles, longs = [], []
        for r in lire_jsonl(source):
            paire, longueur = formater_paire(r, tokenizer, args.max_length, systeme)
            if paire is None:
                longs.append({'id': r['id'], 'tokens_max': longueur})
            else:
                eligibles.append(paire)
        retenus = selectionner(eligibles, quota)
        if len(retenus) != quota:
            raise ValueError('Pas assez de paires éligibles pour le pilote.')
        cible = args.output / f'{split}.jsonl'
        ecrire_immuable(cible, ''.join(json.dumps(r, ensure_ascii=False) + '\n' for r in retenus))
        rapport['splits'][split] = {'eligibles': len(eligibles), 'longs_non_tronques': longs,
            'retenus': len(retenus), 'categories': dict(Counter(r['label_type'] for r in retenus)),
            'ids': [r['id'] for r in retenus], 'source_sha256': hashlib.sha256(source.read_bytes()).hexdigest(),
            'sha256': hashlib.sha256(cible.read_bytes()).hexdigest()}
    ecrire_immuable(args.output / 'manifest.json', json.dumps(rapport, ensure_ascii=False, indent=2) + '\n')
    print(json.dumps({s: {k: v for k, v in r.items() if k not in ('ids', 'longs_non_tronques')}
                      for s, r in rapport['splits'].items()}, indent=2))


if __name__ == '__main__':
    main()


Writing /content/OCR_PROJET_14/training/preparer_dpo.py


In [13]:
%%writefile /content/OCR_PROJET_14/training/entrainer_dpo.py
"""Je poursuis le LoRA SFT avec un pilote DPO TRL, référencé au SFT figé."""

import argparse
import hashlib
from importlib.metadata import version
import json
import os
from pathlib import Path
import sys
import time

RACINE = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(RACINE))
os.environ['MLFLOW_DISABLE_AGENT_HINT'] = '1'

import mlflow
import torch
from datasets import Dataset
from peft import PeftModel, get_peft_model_state_dict
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import DPOConfig, DPOTrainer
from training.entrainer_lora import JournalMLflow, empreinte, lire_jsonl
from training.format_sft import FORMAT_SFT


def empreinte_adaptateur(modele, nom):
    """Vérifie les poids d'un adaptateur, indépendamment de son nom interne."""
    h = hashlib.sha256()
    for cle, poids in sorted(get_peft_model_state_dict(modele, adapter_name=nom).items()):
        h.update(cle.encode())
        h.update(poids.detach().cpu().float().contiguous().numpy().tobytes())
    return h.hexdigest()


def verifier_tokenisation(dataset, source, tokenizer, longueur_max):
    """Contrôle ce que TRL utilise réellement : prompt, réponse et EOS complets."""
    assert len(dataset) == len(source), 'TRL a supprimé des exemples.'
    for ligne, origine in zip(dataset, source):
        prompt = tokenizer.encode(origine['prompt'], add_special_tokens=False)
        assert ligne['prompt_ids'] == prompt, origine['id']
        for cle in ('chosen', 'rejected'):
            attendus = tokenizer.encode(origine['prompt'] + origine[cle], add_special_tokens=False)
            assert prompt + ligne[f'{cle}_ids'] == attendus, origine['id']
            assert attendus[-1] == tokenizer.eos_token_id and len(attendus) <= longueur_max


def main():
    """Trace les données, contrôle la référence puis entraîne 32 pas de DPO."""
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--adapter', type=Path, required=True)
    parser.add_argument('--revision', required=True)
    parser.add_argument('--data-dir', type=Path, required=True)
    parser.add_argument('--output', type=Path, required=True)
    parser.add_argument('--tracking-uri', required=True)
    parser.add_argument('--artifact-location', required=True)
    args = parser.parse_args()
    if args.output.exists() and any(args.output.iterdir()):
        raise ValueError('Le dossier DPO contient déjà un run ; je refuse de l’écraser.')
    args.output.mkdir(parents=True, exist_ok=True)
    origine = json.loads((args.adapter.parent / 'protocole_entrainement.json').read_text())
    assert origine['format_sft'] == FORMAT_SFT and origine['arguments']['revision'] == args.revision
    manifeste = json.loads((args.data_dir / 'manifest.json').read_text())
    assert manifeste['revision'] == args.revision and manifeste['max_length'] == 4096
    donnees = {s: lire_jsonl(args.data_dir / f'{s}.jsonl') for s in ('train', 'validation')}
    for split in donnees:
        assert empreinte(args.data_dir / f'{split}.jsonl') == manifeste['splits'][split]['sha256']
    tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen3-1.7B-Base', revision=args.revision)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = 'left'
    socle = AutoModelForCausalLM.from_pretrained('Qwen/Qwen3-1.7B-Base', revision=args.revision, dtype=torch.bfloat16)
    socle.config.use_cache = False
    modele = PeftModel.from_pretrained(socle, str(args.adapter), is_trainable=True)
    empreinte_initiale = empreinte_adaptateur(modele, 'default')
    config = DPOConfig(
        output_dir=str(args.output / 'checkpoints'), max_steps=32,
        per_device_train_batch_size=2, per_device_eval_batch_size=2,
        gradient_accumulation_steps=8, learning_rate=5e-6, beta=0.1,
        loss_type=['sigmoid'], lr_scheduler_type='cosine', warmup_steps=2,
        max_length=4096, bf16=True, tf32=True, gradient_checkpointing=True,
        precompute_ref_log_probs=True, precompute_ref_batch_size=2,
        eval_strategy='steps', eval_steps=16, save_strategy='steps', save_steps=16,
        save_total_limit=2, load_best_model_at_end=True, metric_for_best_model='eval_loss',
        greater_is_better=False, logging_steps=4, logging_first_step=True,
        report_to='none', seed=3407, data_seed=3407, optim='adamw_torch_fused',
    )
    mlflow.set_tracking_uri(args.tracking_uri)
    client = mlflow.tracking.MlflowClient()
    nom_experience = 'OCR14-Colab-DPO-v3'
    if client.get_experiment_by_name(nom_experience) is None:
        client.create_experiment(nom_experience, artifact_location=args.artifact_location)
    mlflow.set_experiment(nom_experience)
    with mlflow.start_run(run_name='dpo-pilote-256-paires-32-pas') as run:
        debut = time.perf_counter()
        trainer = DPOTrainer(model=modele, args=config, processing_class=tokenizer,
            train_dataset=Dataset.from_list(donnees['train']),
            eval_dataset=Dataset.from_list(donnees['validation']), callbacks=[JournalMLflow()])
        # TRL 1.17.1 copie le LoRA initial dans « ref ». Le socle seul serait une mauvaise référence.
        assert 'ref' in modele.peft_config
        assert empreinte_adaptateur(modele, 'ref') == empreinte_initiale
        assert not any(p.requires_grad for n, p in modele.named_parameters() if '.ref.' in n)
        for split, ds in (('train', trainer.train_dataset), ('validation', trainer.eval_dataset)):
            verifier_tokenisation(ds, donnees[split], tokenizer, 4096)
        protocole = {'format_sft': FORMAT_SFT, 'arguments': vars(args) | {'revision': args.revision},
            'configuration_dpo': config.to_dict(), 'manifeste_donnees': manifeste,
            'reference': 'copie_gelee_du_lora_sft_initial', 'reference_sha256': empreinte_initiale,
            'adapter_source_sha256': empreinte(args.adapter / 'adapter_model.safetensors'),
            'scripts_sha256': {p.name: empreinte(p) for p in (Path(__file__), RACINE / 'training/preparer_dpo.py')},
            'versions': {p: version(p) for p in ('trl', 'transformers', 'peft', 'datasets', 'torch')}}
        chemin = args.output / 'protocole_entrainement.json'
        chemin.write_text(json.dumps(protocole, default=str, ensure_ascii=False, indent=2))
        mlflow.log_params({'revision': args.revision, 'initial_adapter': str(args.adapter),
            'beta': 0.1, 'learning_rate': 5e-6, 'max_steps': 32, 'batch_effectif': 16,
            'train': len(donnees['train']), 'validation': len(donnees['validation']),
            'reference_sha256': empreinte_initiale, 'format_sft': FORMAT_SFT})
        mlflow.log_artifact(str(chemin))
        avant = trainer.evaluate()
        resultat = trainer.train()
        apres = trainer.evaluate()
        assert empreinte_adaptateur(modele, 'ref') == empreinte_initiale, 'La référence a changé.'
        assert empreinte_adaptateur(modele, 'default') != empreinte_initiale, 'Le LoRA n’a pas changé.'
        modele.save_pretrained(args.output / 'adapter', selected_adapters=['default'])
        tokenizer.save_pretrained(args.output / 'adapter')
        trainer.save_state()
        resume = {'mlflow_run_id': run.info.run_id, 'duree_secondes': time.perf_counter() - debut,
            'avant': avant, 'apres': apres, 'train': resultat.metrics,
            'best_model_checkpoint': trainer.state.best_model_checkpoint,
            'reference_inchangee': True, 'tokenisation_verifiee': True}
        chemin_resume = args.output / 'resume_entrainement.json'
        chemin_resume.write_text(json.dumps(resume, ensure_ascii=False, indent=2))
        mlflow.log_artifact(str(chemin_resume))
        print(json.dumps(resume, ensure_ascii=False, indent=2))


if __name__ == '__main__':
    main()


Writing /content/OCR_PROJET_14/training/entrainer_dpo.py


In [14]:
%%writefile /content/OCR_PROJET_14/evaluation/evaluer.py
"""Je recalcule les métriques à partir de réponses de modèle déjà enregistrées."""

import argparse
import hashlib
import json
from pathlib import Path

from metrics import rapport_qa, rapport_triage


RACINE = Path(__file__).resolve().parents[1]
DONNEES = RACINE / "data/processed/sft_deux_phases"


def lire_jsonl(chemin):
    """Charge les objets non vides d'un fichier JSONL."""
    return [json.loads(ligne) for ligne in chemin.read_text().splitlines() if ligne.strip()]


def main():
    """Évalue un fichier de prédictions associé à une phase et un split."""
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("predictions", type=Path)
    parser.add_argument("--phase", choices=("qa", "triage"), required=True)
    parser.add_argument("--split", choices=("validation", "test"), default="test")
    parser.add_argument("--data-dir", type=Path, default=DONNEES)
    parser.add_argument("--limit", type=int, help="Évalue seulement les N premiers exemples du split.")
    parser.add_argument("--output", type=Path)
    args = parser.parse_args()
    prefixe = "phase1_qa" if args.phase == "qa" else "phase2_triage"
    fichier_donnees = args.data_dir / f"{prefixe}_{args.split}.jsonl"
    exemples = lire_jsonl(fichier_donnees)
    if args.limit is not None:
        exemples = exemples[: args.limit]
    predictions = lire_jsonl(args.predictions)
    rapport = (
        rapport_qa(exemples, predictions)
        if args.phase == "qa"
        else rapport_triage(exemples, predictions)
    )
    rapport |= {
        "phase": args.phase,
        "split": args.split,
        "fichier_predictions": str(args.predictions),
        "limite": args.limit,
        "donnees_sha256": hashlib.sha256(fichier_donnees.read_bytes()).hexdigest(),
    }
    sortie = args.output or args.predictions.with_name(args.predictions.stem + "_metriques.json")
    sortie.write_text(json.dumps(rapport, ensure_ascii=False, indent=2) + "\n")
    print(json.dumps(rapport, ensure_ascii=False, indent=2))


if __name__ == "__main__":
    main()


Overwriting /content/OCR_PROJET_14/evaluation/evaluer.py


In [15]:
%%writefile /content/OCR_PROJET_14/evaluation/inference.py
"""Je génère des réponses déterministes pour comparer les checkpoints du projet."""

import argparse
import hashlib
from datetime import datetime, timezone
import json
import os
from pathlib import Path
import sys
import time

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer


RACINE = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(RACINE))
from training.format_sft import construire_prompt
DONNEES = RACINE / "data/processed/sft_deux_phases"
MODELE_BASE = "Qwen/Qwen3-1.7B-Base"


def lire_jsonl(chemin):
    """Charge les objets non vides d'un fichier JSONL."""
    return [json.loads(ligne) for ligne in chemin.read_text().splitlines() if ligne.strip()]


def analyser_arret(ids, fins):
    """Distingue un EOS produit par le modèle d'une coupure à la limite de tokens."""
    for position, identifiant in enumerate(ids):
        if identifiant in fins:
            return {"tokens_sortie": position + 1, "arret": "eos", "token_fin": identifiant}
    return {"tokens_sortie": len(ids), "arret": "limite", "token_fin": None}


def verifier_reprise(predictions, exemples, configuration, taille_lot):
    """Vérifie le préfixe conservé avant d'ajouter des réponses après une interruption."""
    ids = [ligne["id"] for ligne in predictions]
    if len(ids) > len(exemples) or ids != [x["id"] for x in exemples[:len(ids)]] or len(set(ids)) != len(ids):
        raise ValueError("Les identifiants conservés ne correspondent pas au début du corpus.")
    # Les premières sorties locales étaient exclusivement en bfloat16.
    if any(ligne.get(cle, "bfloat16" if cle == "dtype" else None) != valeur
           for ligne in predictions for cle, valeur in configuration.items()):
        raise ValueError("La configuration diffère de celle des réponses conservées.")
    if len(ids) < len(exemples) and len(ids) % taille_lot:
        raise ValueError("Le dernier lot est incomplet ; vérifier le fichier avant de reprendre.")
    return len(ids)


def main():
    """Exécute une inférence batchée et conserve les paramètres avec chaque réponse."""
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--phase", choices=("qa", "triage"), required=True)
    parser.add_argument("--split", choices=("validation", "test"), default="test")
    parser.add_argument("--data-dir", type=Path, default=DONNEES, help="Version figée des partitions à évaluer.")
    parser.add_argument("--model", default=MODELE_BASE)
    parser.add_argument("--revision", help="Révision Hugging Face du socle pour figer les poids.")
    parser.add_argument("--adapter", type=Path, help="Adaptateur LoRA à charger sur le modèle de base.")
    parser.add_argument("--output", type=Path, required=True)
    parser.add_argument("--resume", action="store_true", help="Conserve les lots complets déjà produits.")
    parser.add_argument("--dtype", choices=("bfloat16", "float16"), default="bfloat16",
                        help="float16 permet l'inférence sur un GPU sans bfloat16 natif.")
    parser.add_argument("--limit", type=int)
    parser.add_argument("--batch-size", type=int, default=4)
    parser.add_argument("--max-new-tokens", type=int, default=192)
    parser.add_argument("--stop-token", choices=("historique", "eos"), default="historique",
                        help="eos : arrêt natif, pour le SFT corrigé ; historique : EOS ou im_end.")
    args = parser.parse_args()

    prefixe = "phase1_qa" if args.phase == "qa" else "phase2_triage"
    fichier_donnees = args.data_dir / f"{prefixe}_{args.split}.jsonl"
    exemples = lire_jsonl(fichier_donnees)
    if args.limit:
        exemples = exemples[:args.limit]
    configuration = {
        "modele": args.model,
        "revision": args.revision,
        "adaptateur": str(args.adapter.resolve()) if args.adapter else None,
        "phase": args.phase,
        "split": args.split,
        "generation": {"do_sample": False, "max_new_tokens": args.max_new_tokens},
        "dtype": args.dtype,
    }
    if args.stop_token != "historique":
        configuration["generation"]["stop_token"] = args.stop_token
    if args.data_dir.resolve() != DONNEES.resolve():
        configuration["donnees_sha256"] = hashlib.sha256(fichier_donnees.read_bytes()).hexdigest()
    position_depart = 0
    if args.resume and args.output.exists():
        position_depart = verifier_reprise(
            lire_jsonl(args.output), exemples, configuration, args.batch_size
        )
        if args.output.stat().st_size:
            with args.output.open("rb") as fichier:
                fichier.seek(-1, os.SEEK_END)
                if fichier.read(1) != b"\n":
                    raise ValueError("La dernière ligne n'est pas terminée ; vérifier le JSONL.")
        print(f"Reprise : {position_depart}/{len(exemples)} réponses conservées.", flush=True)
        if position_depart == len(exemples):
            return
    tokenizer = AutoTokenizer.from_pretrained(args.model, revision=args.revision)
    tokenizer.padding_side = "left"
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    modele = AutoModelForCausalLM.from_pretrained(
        args.model,
        revision=args.revision,
        dtype=getattr(torch, args.dtype),
        device_map="auto",
    )
    if args.adapter is not None:
        # Je conserve le tokenizer du socle : seul l'adaptateur change dans la comparaison.
        modele = PeftModel.from_pretrained(modele, str(args.adapter.resolve()), is_trainable=False)
    modele.eval()
    fin_message = tokenizer.convert_tokens_to_ids("<|im_end|>")
    fins = [tokenizer.eos_token_id]
    if args.stop_token == "historique" and isinstance(fin_message, int) and fin_message >= 0:
        fins.append(fin_message)
    args.output.parent.mkdir(parents=True, exist_ok=True)
    debut = time.perf_counter()
    with args.output.open("a" if args.resume else "w", encoding="utf-8") as fichier, torch.inference_mode():
        for position in range(position_depart, len(exemples), args.batch_size):
            lot = exemples[position:position + args.batch_size]
            prompts = [construire_prompt(tokenizer, x["messages"][:2]) for x in lot]
            tokens = tokenizer(
                prompts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=2048,
            ).to(modele.device)
            sorties = modele.generate(
                **tokens,
                max_new_tokens=args.max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=fins,
            )
            longueurs = tokens["attention_mask"].sum(dim=1).tolist()
            for exemple, sortie, longueur in zip(lot, sorties, longueurs):
                # Avec le padding gauche, la génération commence après la largeur commune du batch.
                ids_sortie = sortie[tokens["input_ids"].shape[1]:].tolist()
                arret = analyser_arret(ids_sortie, fins)
                prediction = tokenizer.decode(ids_sortie[:arret["tokens_sortie"]], skip_special_tokens=True).strip()
                fichier.write(json.dumps({
                    "id": exemple["id"],
                    "prediction": prediction,
                    "modele": args.model,
                    "revision": args.revision,
                    "adaptateur": str(args.adapter.resolve()) if args.adapter else None,
                    "dtype": args.dtype,
                    "phase": args.phase,
                    "split": args.split,
                    "generation": configuration["generation"],
                    **arret,
                    "date_utc": datetime.now(timezone.utc).isoformat(),
                    "tokens_entree": int(longueur),
                }, ensure_ascii=False) + "\n")
            fichier.flush()
            os.fsync(fichier.fileno())
            print(f"{min(position + len(lot), len(exemples))}/{len(exemples)}", flush=True)
    print(json.dumps({
        "exemples": len(exemples),
        "secondes": round(time.perf_counter() - debut, 2),
        "sortie": str(args.output),
    }, ensure_ascii=False))


if __name__ == "__main__":
    main()


Overwriting /content/OCR_PROJET_14/evaluation/inference.py


In [16]:
# Je reconstruis les partitions et le pilote, puis je sauvegarde leurs manifestes.
subprocess.run([sys.executable, str(RACINE / 'data_ingestion/figer_partitions.py')], cwd=RACINE, check=True)
subprocess.run([sys.executable, str(RACINE / 'training/preparer_dpo.py'),
                '--revision', REVISION], cwd=RACINE, check=True)
for source in (DATA_V3, PREF_V3):
    shutil.copytree(source, V3 / 'donnees' / source.name, dirs_exist_ok=True)
for relatif in ('data_ingestion/figer_partitions.py', 'evaluation/pilote_expose.json',
                'training/entrainer_lora.py', 'training/format_sft.py',
                'training/preparer_dpo.py', 'training/entrainer_dpo.py',
                'evaluation/inference.py', 'evaluation/evaluer.py', 'evaluation/metrics.py'):
    destination = V3 / 'scripts' / relatif
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(RACINE / relatif, destination)
print('Versions figées et scripts sauvegardés dans Drive.')


Versions figées et scripts sauvegardés dans Drive.


## 16. Reprendre seulement la phase de triage

Je repars du même adaptateur de phase 1. Je garde **357 pas**, batch effectif **16**, learning rate **0,0001**, warmup **20**, seed **3407** et checkpoints tous les **120 pas**. Le retrait des doublons change le nombre d'époques équivalent, pas le budget d'optimisation. Je choisis le checkpoint avec la loss de validation triage, puis je mesure aussi la QA.

Cette reprise fournit une référence nettoyée pour mesurer le DPO. Elle ne cherche pas à optimiser les résultats par de nouveaux essais successifs.


In [17]:
# Je refais seulement le triage, à budget identique : 357 pas, batch effectif 16.
assert torch.cuda.is_available() and torch.cuda.is_bf16_supported()
commande_sft_v3 = [sys.executable, str(RACINE / 'training/entrainer_lora.py'),
    '--revision', REVISION, '--dtype', 'bfloat16', '--phase', 'phase2_triage',
    '--adapter', str(COMPLET / 'adapter'),
    '--train', str(DATA_V3 / 'phase2_triage_train.jsonl'),
    '--validation', str(DATA_V3 / 'phase2_triage_validation.jsonl'),
    '--output', str(SFT_V3), '--run-name', 'phase2-triage-partitions-v3',
    '--max-steps', '357', '--warmup-steps', '20', '--learning-rate', '1e-4',
    '--max-length', '1024', '--batch-size', '8', '--gradient-accumulation', '2',
    '--checkpoint-steps', '120', '--tracking-uri', mlflow.get_tracking_uri(),
    '--experiment', 'OCR14-Colab-SFT-v3', '--artifact-location', (DRIVE / 'suivi/artefacts_sft_v3').as_uri()]
if 'processus_sft_v3' in globals() and processus_sft_v3.poll() is None:
    print('SFT déjà actif.')
elif (SFT_V3 / 'resume_entrainement.json').exists():
    print('SFT v3 déjà terminé.')
else:
    with (V3 / 'sft_execution.log').open('a') as journal:
        processus_sft_v3 = subprocess.Popen(commande_sft_v3, cwd=RACINE,
            stdout=journal, stderr=subprocess.STDOUT, env={**os.environ, 'PYTHONUNBUFFERED': '1'})
    print('SFT v3 lancé :', processus_sft_v3.pid)


SFT v3 lancé : 3373


In [24]:
# Je consulte le processus sans relancer le calcul.
for nom, fichier in (('processus_sft_v3', 'sft_execution.log'), ('processus_dpo_v3', 'dpo_execution.log')):
    if nom in globals():
        etat = globals()[nom].poll()
        print(nom, 'en cours' if etat is None else f'terminé, code {etat}')
        lignes = (V3 / fichier).read_text().replace('\r', '\n').splitlines()
        print('\n'.join([l for l in lignes if l.strip()][-12:]))
        if etat not in (None, 0):
            raise RuntimeError(f'{nom} a échoué : consulter le journal.')


processus_sft_v3 en cours
    "epoch": 3.0254237288135593
  },
  "validation_metrics": {
    "eval_loss": 0.2714458703994751,
    "eval_runtime": 2.3848,
    "eval_samples_per_second": 99.8,
    "eval_steps_per_second": 12.58,
    "epoch": 3.0254237288135593
  },
  "trainable_parameters": 17432576,
  "total_parameters": 1738007552
}


In [25]:
# Même génération que les essais précédents ; seules les partitions et l'adaptateur sont versionnés.
def evaluer_v3(nom, adaptateur):
    """Évalue les deux tâches sur validation et conserve prédictions, métriques et protocole."""
    dossier = EVALUATIONS_V3 / nom
    dossier.mkdir(parents=True, exist_ok=True)
    protocole = {'revision': REVISION, 'dtype': 'bfloat16', 'gpu': torch.cuda.get_device_name(0),
        'batch_size': 32, 'stop_token': 'eos', 'split': 'validation',
        'max_new_tokens': {'qa': 256, 'triage': 192}, 'do_sample': False,
        'adapter_sha256': hashlib.sha256((adaptateur / 'adapter_model.safetensors').read_bytes()).hexdigest(),
        'data_manifest_sha256': hashlib.sha256((DATA_V3 / 'manifest.json').read_bytes()).hexdigest(),
        'scripts_sha256': {p: hashlib.sha256((RACINE / p).read_bytes()).hexdigest() for p in
            ('evaluation/inference.py', 'evaluation/evaluer.py', 'evaluation/metrics.py', 'training/format_sft.py')}}
    fichier_protocole = dossier / 'protocole.json'
    if fichier_protocole.exists():
        assert json.loads(fichier_protocole.read_text()) == protocole, 'Protocole différent.'
    else:
        fichier_protocole.write_text(json.dumps(protocole, indent=2))
    for phase, tokens in (('qa', 256), ('triage', 192)):
        predictions = dossier / f'{phase}_validation.jsonl'
        commande = [sys.executable, str(RACINE / 'evaluation/inference.py'),
            '--phase', phase, '--split', 'validation', '--resume', '--data-dir', str(DATA_V3),
            '--revision', REVISION, '--dtype', 'bfloat16', '--batch-size', '32',
            '--max-new-tokens', str(tokens), '--stop-token', 'eos',
            '--adapter', str(adaptateur), '--output', str(predictions)]
        try:
            with mlflow.start_run(experiment_id=EXPERIENCE_ID, run_name=f'{nom}-{phase}-validation-v3'):
                mlflow.log_params({'modele': nom, 'phase': phase, 'revision': REVISION,
                                   'batch_size': 32, 'dtype': 'bfloat16', 'max_new_tokens': tokens})
                journal = dossier / f'{phase}_execution.log'
                with journal.open('a') as log:
                    subprocess.run(commande, cwd=RACINE, check=True, stdout=log, stderr=subprocess.STDOUT)
                    subprocess.run([sys.executable, str(RACINE / 'evaluation/evaluer.py'), str(predictions),
                        '--phase', phase, '--split', 'validation', '--data-dir', str(DATA_V3)],
                        cwd=RACINE, check=True, stdout=log, stderr=subprocess.STDOUT)
                prefixe = 'phase1_qa' if phase == 'qa' else 'phase2_triage'
                references = [json.loads(l) for l in (DATA_V3 / f'{prefixe}_validation.jsonl').read_text().splitlines()]
                sorties = [json.loads(l) for l in predictions.read_text().splitlines()]
                assert [r['id'] for r in sorties] == [r['id'] for r in references]
                rapport = predictions.with_name(predictions.stem + '_metriques.json')
                mesures = json.loads(rapport.read_text())
                scores = ({'qcm_accuracy': mesures['qcm']['accuracy'],
                           'f1_lexical': mesures['reponses_ouvertes']['f1_lexical_moyen']} if phase == 'qa' else
                          {'macro_f1_priorite': mesures['priorite']['macro_f1'],
                           'esi_analysable': mesures['taux_sortie_esi_valide'],
                           'sous_triage_critique': mesures['taux_sous_triage_critique']})
                scores['taux_eos'] = sum(r['arret'] == 'eos' for r in sorties) / len(sorties)
                mlflow.log_metrics(scores)
                for fichier in (predictions, rapport, journal, fichier_protocole):
                    mlflow.log_artifact(str(fichier))
                print(nom, phase, scores, flush=True)
        finally:
            sauvegarder_suivi()

assert (SFT_V3 / 'resume_entrainement.json').exists(), 'Attendre la fin du SFT.'
evaluer_v3('sft_triage', SFT_V3 / 'adapter')


sft_triage qa {'qcm_accuracy': 0.0382, 'f1_lexical': 0.0359, 'taux_eos': 1.0}
sft_triage triage {'macro_f1_priorite': 0.7715, 'esi_analysable': 1.0, 'sous_triage_critique': 0.021, 'taux_eos': 1.0}


## 17. Vérifier le DPO sur un pilote limité

Le cahier des charges prévoit UltraMedical-Preference. Je conserve ses préférences d'origine, sans les présenter comme validées cliniquement dans ce projet. Elles portent surtout sur des réponses médicales générales en anglais : leur effet sur le triage bilingue reste à mesurer.

| Choix | Pilote |
|---|---|
| Paires | 256 train, 64 validation ; quotas par catégorie de préférence |
| Politique initiale / référence | SFT triage v3 / copie figée du même SFT |
| Optimisation | 32 pas, batch effectif 16, learning rate 0,000005, β = 0,1 |
| Format | Même prompt SFT, EOS natif, réponses complètes |
| Contexte | 4 096 tokens ; 1 paire longue écartée de l'éligibilité, aucune tronquée |
| Sélection | Loss DPO de validation, évaluée aux pas 16 et 32 |

J'utilise [DPOTrainer de TRL](https://huggingface.co/docs/trl/dpo_trainer), version 1.17.1. Le pilote vérifie les tokens réellement utilisés, l'identité initiale des deux adaptateurs, l'immobilité de la référence et une mise à jour effective du LoRA entraîné. Un test d'intégration sur un modèle miniature CPU a vérifié ce mécanisme avant le calcul GPU.

Le DPO apprend à augmenter la préférence relative pour `chosen` face à `rejected`, par rapport au SFT figé. La loss et les récompenses relatives mesurent cet objectif ; elles ne mesurent pas directement la justesse médicale. Je conserve donc les évaluations QA et triage, avec une attention particulière au rappel des urgences et au sous-triage critique. Je ne génère aucune prédiction du test final pendant ce pilote.


In [26]:
# Je ne lance le pilote DPO qu'après un SFT terminé et ses deux évaluations.
assert (SFT_V3 / 'resume_entrainement.json').exists()
assert (EVALUATIONS_V3 / 'sft_triage/triage_validation_metriques.json').exists()
commande_dpo_v3 = [sys.executable, str(RACINE / 'training/entrainer_dpo.py'),
    '--adapter', str(SFT_V3 / 'adapter'), '--revision', REVISION,
    '--data-dir', str(PREF_V3), '--output', str(DPO_V3),
    '--tracking-uri', mlflow.get_tracking_uri(), '--artifact-location', (DRIVE / 'suivi/artefacts_dpo_v3').as_uri()]
if 'processus_dpo_v3' in globals() and processus_dpo_v3.poll() is None:
    print('DPO déjà actif.')
elif (DPO_V3 / 'resume_entrainement.json').exists():
    print('Pilote DPO déjà terminé.')
else:
    with (V3 / 'dpo_execution.log').open('a') as journal:
        processus_dpo_v3 = subprocess.Popen(commande_dpo_v3, cwd=RACINE,
            stdout=journal, stderr=subprocess.STDOUT, env={**os.environ, 'PYTHONUNBUFFERED': '1'})
    print('Pilote DPO lancé :', processus_dpo_v3.pid)


Pilote DPO lancé : 5576


In [29]:
# Je consulte le processus sans relancer le calcul.
for nom, fichier in (('processus_sft_v3', 'sft_execution.log'), ('processus_dpo_v3', 'dpo_execution.log')):
    if nom in globals():
        etat = globals()[nom].poll()
        print(nom, 'en cours' if etat is None else f'terminé, code {etat}')
        lignes = (V3 / fichier).read_text().replace('\r', '\n').splitlines()
        print('\n'.join([l for l in lignes if l.strip()][-12:]))
        if etat not in (None, 0):
            raise RuntimeError(f'{nom} a échoué : consulter le journal.')


processus_sft_v3 terminé, code 0
    "epoch": 3.0254237288135593
  },
  "validation_metrics": {
    "eval_loss": 0.2714458703994751,
    "eval_runtime": 2.3848,
    "eval_samples_per_second": 99.8,
    "eval_steps_per_second": 12.58,
    "epoch": 3.0254237288135593
  },
  "trainable_parameters": 17432576,
  "total_parameters": 1738007552
}
processus_dpo_v3 terminé, code 0
  "train": {
    "train_runtime": 69.3995,
    "train_samples_per_second": 7.378,
    "train_steps_per_second": 0.461,
    "total_flos": 6441039886540800.0,
    "train_loss": 0.5561044812202454,
    "epoch": 2.0
  },
  "best_model_checkpoint": "/content/drive/MyDrive/OCR_PROJET_14/experiences/sft_dpo_v3_20261105/dpo_pilote/checkpoints/checkpoint-32",
  "reference_inchangee": true,
  "tokenisation_verifiee": true
}


In [30]:
assert (DPO_V3 / 'resume_entrainement.json').exists(), 'Attendre la fin du pilote DPO.'
evaluer_v3('dpo_pilote', DPO_V3 / 'adapter')


dpo_pilote qa {'qcm_accuracy': 0.0382, 'f1_lexical': 0.0363, 'taux_eos': 1.0}
dpo_pilote triage {'macro_f1_priorite': 0.7674, 'esi_analysable': 1.0, 'sous_triage_critique': 0.021, 'taux_eos': 1.0}


## 18. Comparer les effets du DPO avant de choisir le modèle

Je compare le SFT triage et son prolongement DPO sur les **mêmes exemples de validation**, avec le même décodage. Je ne consulte pas le test pour choisir un entraînement.

| Question | Indicateur | Limite d'interprétation |
|---|---|---|
| Les préférences sont-elles apprises ? | Perte DPO, marge et accuracy des récompenses | Mesure relative à la référence SFT, pas une accuracy médicale |
| Le triage est-il préservé ? | Macro-F1, rappel de l'urgence maximale, sous-triage critique | Cibles issues d'un consensus LLM, sans validation clinique |
| Le français et l'anglais restent-ils cohérents ? | Accord des paires et scores par langue | Les traductions ne sont pas des observations indépendantes |
| La réponse générale est-elle conservée ? | Accuracy QCM et F1 lexical QA | Le F1 lexical ne mesure ni la factualité ni l'absence d'hallucination |

Le pilote porte sur **256 paires d'entraînement et 64 de validation**, pendant 32 pas. Je vérifie d'abord la chaîne technique et les régressions ; je ne présente pas cet essai comme un DPO sur l'ensemble du corpus.


In [31]:
from evaluation.metrics import extraire_esi

def lire_lignes(chemin):
    """Lit les exemples ou prédictions enregistrés, sans nouvel appel au modèle."""
    return [json.loads(l) for l in chemin.read_text().splitlines() if l.strip()]

references_triage = lire_lignes(DATA_V3 / 'phase2_triage_validation.jsonl')
bilan_v3 = {'partitions': json.loads((DATA_V3 / 'manifest.json').read_text()),
            'pilote_dpo': json.loads((PREF_V3 / 'manifest.json').read_text()),
            'modeles': {}, 'entrainements': {}}
comparaison_v3 = []
revue_v3 = {}
for nom, repertoire in (('sft_triage', SFT_V3), ('dpo_pilote', DPO_V3)):
    dossier = EVALUATIONS_V3 / nom
    rapports = {phase: json.loads((dossier / f'{phase}_validation_metriques.json').read_text())
                for phase in ('qa', 'triage')}
    predictions = {phase: lire_lignes(dossier / f'{phase}_validation.jsonl') for phase in ('qa', 'triage')}
    par_id = {x['id']: x for x in predictions['triage']}
    critiques = []
    for exemple in references_triage:
        sortie = par_id[exemple['id']]
        esi = extraire_esi(sortie['prediction'])
        if exemple['esi'] <= 2 and esi is not None and esi >= 3:
            critiques.append({'id': exemple['id'], 'id_paire': exemple.get('id_paire'),
                'langue': exemple['langue'], 'esi_reference': exemple['esi'], 'esi_predit': esi,
                'cas': exemple['messages'][1]['content'], 'prediction': sortie['prediction']})
    urgents = sum(x['esi'] <= 2 for x in references_triage)
    rapports['comportement'] = {p: {'nombre': len(xs), 'arrets_eos': sum(x['arret'] == 'eos' for x in xs)}
                               for p, xs in predictions.items()}
    rapports['sous_triage_critique_detail'] = {'nombre': len(critiques), 'urgents_reference': urgents,
        'proportion_parmi_urgents': len(critiques) / urgents, 'ids': [x['id'] for x in critiques]}
    bilan_v3['modeles'][nom] = rapports
    bilan_v3['entrainements'][nom] = json.loads((repertoire / 'resume_entrainement.json').read_text())
    revue_v3[nom] = critiques
    comparaison_v3.append({'modele': nom, 'macro_F1_priorite': rapports['triage']['priorite']['macro_f1'],
        'rappel_maximale': rapports['triage']['priorite']['rappel_par_classe']['maximale'],
        'accuracy_ESI': rapports['triage']['esi']['accuracy'],
        'ESI_analysable': rapports['triage']['taux_sortie_esi_valide'],
        'sous_triages_critiques': len(critiques), 'urgents_reference': urgents,
        'accord_bilingue': rapports['triage']['coherence_bilingue_esi']['taux_meme_prediction'],
        'accuracy_QCM': rapports['qa']['qcm']['accuracy'],
        'F1_lexical_QA': rapports['qa']['reponses_ouvertes']['f1_lexical_moyen']})
ids_sft = set(bilan_v3['modeles']['sft_triage']['sous_triage_critique_detail']['ids'])
ids_dpo = set(bilan_v3['modeles']['dpo_pilote']['sous_triage_critique_detail']['ids'])
bilan_v3['evolution_erreurs_critiques'] = {'persistantes': sorted(ids_sft & ids_dpo),
    'nouvelles': sorted(ids_dpo - ids_sft), 'corrigees': sorted(ids_sft - ids_dpo)}
for nom, contenu in (('bilan_v3.json', bilan_v3), ('revue_sous_triage_v3.json', revue_v3)):
    (V3 / nom).write_text(json.dumps(contenu, ensure_ascii=False, indent=2))
with mlflow.start_run(experiment_id=EXPERIENCE_ID, run_name='bilan-sft-dpo-v3'):
    for nom in ('bilan_v3.json', 'revue_sous_triage_v3.json'):
        mlflow.log_artifact(str(V3 / nom))
sauvegarder_suivi()
display(pd.DataFrame(comparaison_v3).set_index('modele'))
print('Erreurs critiques :', bilan_v3['evolution_erreurs_critiques'])
print('DPO avant/après :', bilan_v3['entrainements']['dpo_pilote']['avant'],
      bilan_v3['entrainements']['dpo_pilote']['apres'])


,macro_F1_priorite,rappel_maximale,accuracy_ESI,ESI_analysable,sous_triages_critiques,urgents_reference,accord_bilingue,accuracy_QCM,F1_lexical_QA
modele,,,,,,,,,
sft_triage,0.7715,0.95,0.7311,1.0,5,100,0.7395,0.0382,0.0359
dpo_pilote,0.7674,0.95,0.7269,1.0,5,100,0.7647,0.0382,0.0363


Erreurs critiques : {'persistantes': ['mediqal-212e134bd4c608aea485dbdc531167c42366f3c75975d0805244602dab710643', 'mediqal-212e134bd4c608aea485dbdc531167c42366f3c75975d0805244602dab710643-en', 'mediqal-5969c60d532801e8e3351c2ae713ffae87fd9988040b55f32c32cc7a6d8df97e', 'mediqal-5969c60d532801e8e3351c2ae713ffae87fd9988040b55f32c32cc7a6d8df97e-en', 'ultramedical-405a576ff747e79d062fb7dd72d15a94d36474ac9aaffddab5495153d54c0633-fr'], 'nouvelles': [], 'corrigees': []}
DPO avant/après : {'eval_loss': 0.6931471824645996, 'eval_model_preparation_time': 0.0146, 'eval_runtime': 1.9877, 'eval_samples_per_second': 32.198, 'eval_steps_per_second': 16.099, 'eval_entropy': 0.9923112839460373, 'eval_num_tokens': 0.0, 'eval_logits/chosen': 5.285970841119311, 'eval_logits/rejected': 5.195537672059531, 'eval_mean_token_accuracy': 0.6708096638321877, 'eval_rewards/chosen': 0.0, 'eval_rewards/rejected': 0.0, 'eval_rewards/accuracies': 0.0, 'eval_rewards/margins': 0.0, 'eval_logps/chosen': -472.5694689750671

## 19. Résultats du pilote et décision pour la suite

### Ce que j'observe

Le SFT v3 dure **222 s** (entraînement et validation finale). Le DPO dure **100 s** (préparation, entraînement et évaluations de préférence), dont 69 s d'optimisation. Ces durées ne comprennent pas l'installation ni toutes les générations comparatives.

| Validation triage — 238 entrées | SFT v3 | Après DPO pilote |
|---|---:|---:|
| Macro-F1 des priorités | **0,7715** | 0,7674 |
| Rappel de l'urgence maximale | 95/100 | 95/100 |
| Sous-triages critiques | 5/100 urgents | Les mêmes 5/100 |
| Accord ESI français / anglais | 88/119 paires | 91/119 paires |
| Macro-F1 — français | **0,8180** | 0,7991 |
| Macro-F1 — anglais | 0,7229 | **0,7349** |

Toutes les sorties triage sont analysables et terminées par EOS. Le DPO modifie seulement **3 décisions ESI sur 238** : une devient correcte en anglais et deux deviennent incorrectes en français. L'écart global est faible ; cet essai unique ne démontre pas une dégradation statistiquement significative.

Sur les 64 préférences de validation, la perte DPO passe de **0,6931 à 0,5361** et 51 paires ont une marge de récompense positive (79,7 %). Au départ, les récompenses sont identiques puisque politique et référence sont identiques : l'accuracy relative à zéro n'est donc **pas** une accuracy médicale initiale de 0 %.

### Ce que les réponses révèlent

- Les cinq erreurs critiques concernent **trois cas distincts** : palpitations avec cardiopathie ischémique (FR/EN), vulvite très douloureuse avec céphalée et photophobie (FR/EN), douleur testiculaire intense (FR). Les deux modèles prédisent ESI 3, contre ESI 2 dans le consensus. Je conserve ces désaccords, sans modifier les cibles pour améliorer les scores.
- Dans le cas de vulvite, les deux modèles écrivent « infection fébrile », alors que le texte ne mentionne pas de fièvre. L'avertissement médical présent dans la réponse ne suffit donc pas à garantir une justification fidèle.
- Le format triage déborde sur la QA : la première question MedQuAD reçoit une classification ESI. L'accuracy QCM reste **5/131 (3,82 %)** ; le F1 lexical anglais reste très faible (**0,0359 → 0,0363**). Ce pilote ne restaure pas la capacité de réponse générale.

### Ce que j'en retiens

```text
Objectif de préférence amélioré ≠ meilleur triage
Accord entre langues accru      ≠ meilleure exactitude dans chaque langue
Réponse bien formatée           ≠ justification fidèle au cas
```

Je conserve **le SFT triage v3 comme candidat principal du POC** et le DPO comme variante expérimentale. Le rappel global de 95 % masque un rappel français de 94 % ; les deux langues d'un cas sont dépendantes. Je ne transforme pas ce résultat de validation en garantie clinique.

Je n'étends pas automatiquement le DPO aux 4 470 paires : l'essai n'apporte pas de gain sur le critère métier principal. La suite porte sur le POC (API, questionnaire, traçabilité et tests), puis sur une comparaison finale avec les **226 entrées de test triage restantes**, une fois les choix figés. Si je reprends le DPO, je définirai un nouvel essai justifié par l'adéquation des préférences au triage, plutôt que par le seul volume.

La cellule de clôture a confirmé la présence des adaptateurs, protocoles, réponses et traces MLflow dans le montage Drive avant la libération de la VM. Ce contrôle ne prouve toutefois pas que tous les fichiers étaient déjà synchronisés côté Drive ; je vérifie leur récupération dans la section suivante. Les 1 000 prédictions de cette comparaison ont été récupérées localement et les quatre rapports de métriques reproduits à l'identique, sans nouvelle inférence. Les fichiers de référence sont `data/evaluation/colab_sft_dpo_v3/bilan_v3.json` et `revue_sous_triage_v3.json`.


In [ ]:
# Je libère le GPU après les évaluations et la sauvegarde sur Drive.
from google.colab import runtime
for nom in ('processus_sft_v3', 'processus_dpo_v3'):
    assert nom not in globals() or globals()[nom].poll() == 0, nom
sauvegarder_suivi()
for chemin in (BASE_DRIVE, V3 / 'bilan_v3.json', V3 / 'revue_sous_triage_v3.json',
               SFT_V3 / 'adapter/adapter_model.safetensors', DPO_V3 / 'adapter/adapter_model.safetensors'):
    assert chemin.exists() and chemin.stat().st_size > 0, chemin
print('Poids, résultats et suivi sauvegardés. Libération de la VM.', flush=True)
# Je termine les écritures distantes avant de supprimer la VM.
from google.colab import drive
drive.flush_and_unmount(timeout_ms=300_000)
runtime.unassign()


## 20. Retrouver les expériences en local

```text
Drive / OCR_PROJET_14
        ↓ copie des fichiers + inventaire + empreintes SHA256
 data/colab_archive/original/      ← originaux conservés
        ↓ copie SQLite, URI d’artefacts adaptées
 data/colab_archive/local/         ← consultation MLflow locale
```

Je conserve le notebook Colab téléchargé dans l’archive. Le présent notebook contient déjà des sorties et analyses plus récentes : je ne le remplace pas par une version antérieure.

Depuis la racine du dépôt, `python evaluation/restaurer_colab.py` vérifie les fichiers et prépare le suivi local. Je consulte ensuite MLflow avec :

```bash
.venv/bin/mlflow ui --host 127.0.0.1 --port 5001 \
  --backend-store-uri sqlite:///data/colab_archive/local/mlflow_colab.db
```

Les paramètres historiques gardent leurs chemins Colab pour la provenance ; les liens d’artefacts pointent vers la copie locale. Cette récupération ne lance aucun entraînement. Je distingue la présence des fichiers, leur intégrité et leur exhaustivité : une copie valide d’une sauvegarde ancienne ne prouve pas que toute l’expérience finale est archivée.

Je sécurise les prochaines fermetures avec `drive.flush_and_unmount()` avant `runtime.unassign()`. En cas d’échec de synchronisation, la cellule s’arrête sans libérer la VM. Cette correction est préparée, pas exécutée ici. [Fonction officielle Colab](https://github.com/googlecolab/colabtools/blob/main/google/colab/drive.py).

### Constat de récupération

- **363 fichiers, 2,82 Go** : tailles et SHA256 vérifiés ; fichiers JSON/JSONL et archive ZIP contrôlés.
- **28 exécutions MLflow** dans la copie distante : 26 terminées et 2 échouées. Tous les dossiers d’artefacts des exécutions terminées sont présents localement.
- Le LoRA SFT sélectionné correspond à l’empreinte utilisée par le Space. Les checkpoints sont également accessibles par `models/colab_experiences/`.
- **Limite :** les poids finaux DPO, le contenu complet de `checkpoint-32` et le suivi MLflow post-DPO ne sont pas récupérés. La base distante accessible est datée du 5 novembre 2026 à 11:49:55 UTC. Les résultats DPO déjà présents dans `data/evaluation/colab_sft_dpo_v3` restent conservés. Je ne conclus pas à une perte définitive ; la connexion Colab est à rétablir pour vérifier la dernière version.

Les preuves et les limites sont réunies dans `data/colab_archive/verification.json`. Aucun entraînement ni GPU Colab n’a été relancé pour cette copie.


## 21. Mesurer le candidat après la sélection

Je conserve le SFT v3 comme candidat principal. Le [notebook 05](05_finalisation_colab.ipynb) rassemble le test final figé et la vérification temporaire de vLLM. Je mesure le socle et le SFT sur 262 QA et 226 triages ; je ne mesure le DPO que si ses poids originaux sont récupérés et vérifiés.

```text
choix sur validation → protocole figé → test indépendant → bilan
```

Le protocole est enregistré dans `protocole_test_final.json`. Les scripts refusent les partitions ou poids différents. Aucun résultat de cette dernière exécution n’est encore confirmé ; je ne remplace pas les mesures de validation par des résultats supposés.
